In [20]:
# ============================================================
# 01_reliability_analysis_figures.py
#
# 目的：
# 1. 不需要既有 detail CSV
# 2. 重新 evaluate Original 與 Proposed 兩個模型
# 3. 產生一份必要 CSV：reliability_required_fields.csv
# 4. 產生所有「兩模型比較版」圖表
# ============================================================

import os
import re
import json
import ast
import math
import textwrap
from copy import deepcopy
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torchvision
from tqdm import tqdm

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

from dataset import UIUCPolyvoreRetrievalDataset
from outfit_transformer import (
    OutfitTransformerRetrieval,
    LinearImageEncoder,
    LinearTextEncoder,
)
from config.cir_cond_hardneg import *


# ============================================================
# 0. WSL / Jupyter：直接使用 Windows 中文字型
# ============================================================

FONT_CANDIDATES = [
    "/mnt/c/Windows/Fonts/msjh.ttc",       # 微軟正黑體
    "/mnt/c/Windows/Fonts/msjhbd.ttc",     # 微軟正黑體 Bold
    "/mnt/c/Windows/Fonts/kaiu.ttf",       # 標楷體
    "/mnt/c/Windows/Fonts/mingliu.ttc",    # 新細明體 / 細明體
]

FONT_PATH = None
for path in FONT_CANDIDATES:
    if os.path.exists(path):
        FONT_PATH = path
        break

if FONT_PATH is None:
    raise FileNotFoundError(
        "找不到 Windows 中文字型。請確認 WSL 可以讀到 /mnt/c/Windows/Fonts/。"
    )

font_manager.fontManager.addfont(FONT_PATH)
font_name = font_manager.FontProperties(fname=FONT_PATH).get_name()

matplotlib.rcParams["font.family"] = font_name
matplotlib.rcParams["font.sans-serif"] = [font_name]
matplotlib.rcParams["axes.unicode_minus"] = False

print("中文字型設定完成：", font_name)
print("使用字型路徑：", FONT_PATH)


# ============================================================
# 1. 路徑與模型設定
# ============================================================

SEED = 1

RUN_CONFIGS = [
    {
        "run_tag": "original",
        "model_tag": "cir_old",
        "feat_tag": "old",
        "model_name": "Original CIR / outfitUrlTitle",
    },
    {
        "run_tag": "proposed",
        "model_tag": "cir_new",
        "feat_tag": "new",
        "model_name": "Proposed CIR / NewoutfitUrlTitle",
    },
]

BASELINE_TAG = "original"
PROPOSED_TAG = "proposed"

EXP_ROOT = "./experiments"

OUTPUT_DIR = "./reliability_outputs/compare"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 只清掉本次會重產生的檔案，避免舊圖誤放進 dashboard
for fname in os.listdir(OUTPUT_DIR):
    if fname.startswith("plot_") and fname.endswith(".png"):
        os.remove(os.path.join(OUTPUT_DIR, fname))

for fname in ["reliability_required_fields.csv", "reliability_dashboard.html"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        os.remove(fpath)

# metadata 來源
FRAG_JSONL = "/home/ester/valid_description/wos_split_results_v5_merged_retry_round3.jsonl"
ORIGINAL_TITLE_JSON = "/home/ester/valid_description/polyvore_data/polyvore_outfits/polyvore_outfit_titles.json"
CATEGORY_CSV = "/home/ester/valid_description/polyvore_data/polyvore_outfits/categories.csv"

POLYVORE_SPLIT = "disjoint"

MAX_DISTRACTORS = 3000
MAX_EVAL_SAMPLES = None        # 測試時可設 100；正式請用 None

HIGH_CONF_THRESHOLD = 0.85
CONFIDENCE_METHOD = "gap_sigmoid"   # gap_sigmoid / softmax / hybrid
SOFTMAX_TEMPERATURE = 0.05
GAP_SCALE = 50.0
N_BINS = 10


# ============================================================
# 2. 工具函式
# ============================================================

def normalize_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):
        return " | ".join(str(i).strip() for i in x if str(i).strip())
    return str(x).strip()


def normalize_token_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"[_/|,-]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def parse_list_like(x):
    if isinstance(x, list):
        return x

    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return []

        try:
            v = json.loads(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass

        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass

    return []


def to_safe_id(x):
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass

    try:
        return str(int(float(x)))
    except Exception:
        return str(x)


def to_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def pct(x):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:.2f}%"


def wrap_text(s, width=24):
    s = "" if pd.isna(s) else str(s)
    return "\n".join(textwrap.wrap(s, width=width))


def sigmoid(x):
    if x >= 0:
        z = math.exp(-x)
        return 1 / (1 + z)
    z = math.exp(x)
    return z / (1 + z)


def assign_confidence_band(x):
    x = to_float(x)

    if pd.isna(x):
        return "unknown"
    if x < 0.60:
        return "0.50–0.60"
    if x < 0.70:
        return "0.60–0.70"
    if x < 0.80:
        return "0.70–0.80"
    if x < 0.90:
        return "0.80–0.90"
    return "0.90–1.00"


# ============================================================
# 3. metadata 讀取與 subgroup 切分
# ============================================================

def load_original_titles(path):
    if not os.path.exists(path):
        print(f"[WARN] 找不到 original title JSON：{path}")
        return pd.DataFrame(columns=["set_id", "original_text"])

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    for sid, value in data.items():
        if isinstance(value, str):
            text = value.strip()
        elif isinstance(value, dict):
            url = normalize_text(value.get("url_name", value.get("url", "")))
            title = normalize_text(value.get("title", ""))
            text = " ".join(p for p in [url, title] if p).strip()
        else:
            text = normalize_text(value)

        rows.append({
            "set_id": str(sid),
            "original_text": text,
        })

    return pd.DataFrame(rows)


def load_fragments(path):
    if not os.path.exists(path):
        print(f"[WARN] 找不到 FRAG_JSONL：{path}")
        return pd.DataFrame(columns=[
            "set_id",
            "title_full_text",
            "weather_terms",
            "occasion_terms",
            "style_terms",
            "weather_text",
            "occasion_text",
            "style_text",
        ])

    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            obj = json.loads(line)

            if obj.get("valid") is not True:
                continue

            frags = obj.get("fragments", {})

            weather_terms = frags.get("weather", [])
            occasion_terms = frags.get("occasion", [])
            style_terms = frags.get("style", [])

            if not isinstance(weather_terms, list):
                weather_terms = []
            if not isinstance(occasion_terms, list):
                occasion_terms = []
            if not isinstance(style_terms, list):
                style_terms = []

            rows.append({
                "set_id": str(obj.get("id", "")),
                "title_full_text": normalize_text(obj.get("title", "")),
                "weather_terms": weather_terms,
                "occasion_terms": occasion_terms,
                "style_terms": style_terms,
                "weather_text": normalize_text(weather_terms),
                "occasion_text": normalize_text(occasion_terms),
                "style_text": normalize_text(style_terms),
            })

    return pd.DataFrame(rows)


def load_category_map(path):
    if not os.path.exists(path):
        print(f"[WARN] 找不到 categories.csv：{path}")
        return pd.DataFrame(columns=["category_id", "fine_category", "major_category"])

    raw = pd.read_csv(path, header=None, dtype=str).fillna("")

    if raw.empty:
        return pd.DataFrame(columns=["category_id", "fine_category", "major_category"])

    out = pd.DataFrame()
    out["category_id"] = raw.iloc[:, 0].astype(str).str.strip()

    if raw.shape[1] >= 2:
        out["fine_category"] = raw.iloc[:, 1].astype(str).apply(normalize_token_text)
    else:
        out["fine_category"] = ""

    if raw.shape[1] >= 3:
        out["major_category"] = raw.iloc[:, 2].astype(str).apply(normalize_token_text)
    else:
        out["major_category"] = ""

    out = out.drop_duplicates(subset=["category_id"], keep="first").reset_index(drop=True)

    return out


TEMP_PATTERN = re.compile(r"(-?\d+(?:\.\d+)?)\s*°?\s*([CFcf])")

def extract_temperature_celsius_from_text(text):
    if not isinstance(text, str) or not text.strip():
        return np.nan

    m = TEMP_PATTERN.search(text)

    if not m:
        return np.nan

    value = float(m.group(1))
    unit = m.group(2).upper()

    if unit == "F":
        value = (value - 32.0) * 5.0 / 9.0

    return value


def extract_row_temperature(row):
    candidates = [
        row.get("weather_text", ""),
        row.get("title_full_text", ""),
        row.get("original_text", ""),
    ]

    for txt in candidates:
        temp = extract_temperature_celsius_from_text(txt)
        if pd.notna(temp):
            return temp

    return np.nan


FORMAL_KEYWORDS = r"\b(formal|office|business|work|professional|interview|meeting|wedding|party|cocktail|elegant|tailored|blazer|suit)\b"
CASUAL_KEYWORDS = r"\b(casual|weekend|streetwear|street|relaxed|everyday|daily|chill|laid-back|lounge|vacation|travel)\b"

def assign_occasion_group(row):
    text = " ".join([
        normalize_text(row.get("occasion_text", "")),
        normalize_text(row.get("title_full_text", "")),
        normalize_text(row.get("original_text", "")),
    ]).lower()

    if re.search(FORMAL_KEYWORDS, text):
        return "formal"

    if re.search(CASUAL_KEYWORDS, text):
        return "casual"

    return "unknown"


def get_style_count(x):
    if isinstance(x, list):
        return len([t for t in x if str(t).strip()])
    return 0


CLOTHING_MAJOR = {
    "all body",
    "bottoms",
    "tops",
    "outerwear",
}

ACCESSORY_MAJOR = {
    "bags",
    "shoes",
    "accessories",
    "hats",
    "jewellery",
    "jewelry",
    "scarves",
    "sunglasses",
}

def assign_category_group(x):
    x = normalize_token_text(x)

    if x in CLOTHING_MAJOR:
        return "clothing-led"

    if x in ACCESSORY_MAJOR:
        return "accessory-led"

    return "unknown"


# ============================================================
# 4. 資料集與模型
# ============================================================

def resolve_paths():
    current_dir = os.getcwd()
    parent_dir = os.path.dirname(current_dir)

    feat_dir = os.path.join(parent_dir, "fashionclip_data")
    data_root = os.path.join(parent_dir, "polyvore_data", "polyvore_outfits")
    final_datadir = os.path.join(parent_dir, "polyvore_data")

    img_feat_path = os.path.join(feat_dir, "img_feats_fashionClip.pkl")
    txt_feat_path = os.path.join(
        feat_dir,
        "encoded_title_description_distiluse-base-multilingual-cased-v2.pkl"
    )
    category_feat_path = os.path.join(
        feat_dir,
        "encoded_category_distiluse-base-multilingual-cased-v2.pkl"
    )

    return parent_dir, feat_dir, data_root, final_datadir, img_feat_path, txt_feat_path, category_feat_path


def get_outfit_feat_path(feat_dir, feat_tag):
    if feat_tag == "new":
        return os.path.join(feat_dir, "encoded_NewoutfitUrlTitle_en_fashionClip.pkl")
    if feat_tag == "old":
        return os.path.join(feat_dir, "encoded_outfitUrlTitle_en_fashionClip.pkl")
    raise ValueError(f"Unknown feat_tag: {feat_tag}")


def build_transform():
    normalize = torchvision.transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    return torchvision.transforms.Compose([
        torchvision.transforms.Resize((img_size, img_size)),
        torchvision.transforms.ToTensor(),
        normalize,
    ])


def build_datasets(run_cfg):
    parent_dir, feat_dir, data_root, final_datadir, img_feat_path, txt_feat_path, category_feat_path = resolve_paths()

    outfit_feat_path = get_outfit_feat_path(
        feat_dir,
        run_cfg["feat_tag"]
    )

    split_dir = os.path.join(data_root, POLYVORE_SPLIT)

    if not os.path.exists(split_dir):
        raise FileNotFoundError(f"找不到 split 資料夾：{split_dir}")

    transform = build_transform()

    train_dat = UIUCPolyvoreRetrievalDataset(
        datadir=final_datadir,
        txt_feat_path=txt_feat_path,
        img_feat_path=img_feat_path,
        category_feat_path=category_feat_path,
        outfit_feat_path=outfit_feat_path,
        polyvore_split=POLYVORE_SPLIT,
        split="train",
        max_item_len=max_item_len,
        transform=transform,
        sample_hard_negative=False,
    )

    test_dat = UIUCPolyvoreRetrievalDataset(
        datadir=final_datadir,
        txt_feat_path=txt_feat_path,
        img_feat_path=img_feat_path,
        category_feat_path=category_feat_path,
        outfit_feat_path=outfit_feat_path,
        polyvore_split=POLYVORE_SPLIT,
        split="test",
        max_item_len=max_item_len,
        transform=transform,
        sample_hard_negative=False,
    )

    return train_dat, test_dat


def build_model(run_cfg, device):
    model = OutfitTransformerRetrieval(
        img_encoder=LinearImageEncoder(
            img_inp_size=img_inp_size,
            img_emb_size=img_emb_size,
        ),
        text_encoder=LinearTextEncoder(
            txt_inp_size=txt_inp_size,
            txt_emb_size=txt_emb_size,
        ),
        outfit_txt_encoder=LinearTextEncoder(
            txt_inp_size=outfit_txt_inp_size,
            txt_emb_size=txt_emb_size + img_emb_size,
        ),
        nhead=nhead,
        num_layers=num_layers,
        margin=margin,
        target_item_info="category",
        use_outfit_txt=use_outfit_txt,
    )

    ckpt_path = os.path.join(
        EXP_ROOT,
        f"{run_cfg['model_tag']}_seed{SEED}",
        "ckpt.pt"
    )

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"找不到 checkpoint：{ckpt_path}")

    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    model = model.to(device)
    model.eval()

    print(f"Checkpoint loaded for {run_cfg['run_tag']}:", ckpt_path)

    return model


# ============================================================
# 5. 模型 evaluate
# ============================================================

def build_distractors(train_dat, test_dat, model, device, ctx):
    distractors_id = {}
    distractors_feats = {}
    distractors_seen = {}

    for k_fg, v in tqdm(test_dat.fg2ims.items(), desc="Build distractors from test"):
        distractors_id[k_fg] = []
        distractors_feats[k_fg] = []
        distractors_seen[k_fg] = set()

        for _, item_lst in v.items():
            for item_id in item_lst:
                if item_id not in distractors_seen[k_fg]:
                    distractors_seen[k_fg].add(item_id)
                    distractors_id[k_fg].append(item_id)

                    emb = test_dat.extract_emb(
                        {"item_id": item_id},
                        model,
                        device,
                        ctx,
                        to_numpy=True,
                    )
                    distractors_feats[k_fg].append(emb)

                if len(distractors_id[k_fg]) >= MAX_DISTRACTORS:
                    break

            if len(distractors_id[k_fg]) >= MAX_DISTRACTORS:
                break

    for k_fg, v in tqdm(train_dat.fg2ims.items(), desc="Fill distractors from train"):
        if k_fg not in distractors_id:
            continue

        for _, item_lst in v.items():
            for item_id in item_lst:
                if item_id not in distractors_seen[k_fg]:
                    distractors_seen[k_fg].add(item_id)
                    distractors_id[k_fg].append(item_id)

                    emb = test_dat.extract_emb(
                        {"item_id": item_id},
                        model,
                        device,
                        ctx,
                        to_numpy=True,
                    )
                    distractors_feats[k_fg].append(emb)

                if len(distractors_id[k_fg]) >= MAX_DISTRACTORS:
                    break

            if len(distractors_id[k_fg]) >= MAX_DISTRACTORS:
                break

    distractors_id = {
        k: np.array(v)
        for k, v in distractors_id.items()
        if len(v) >= MAX_DISTRACTORS
    }

    distractors_feats = {
        k: torch.tensor(np.concatenate(v)).float()
        for k, v in distractors_feats.items()
        if len(v) >= MAX_DISTRACTORS
    }

    print(f"Selected fine-grained categories: {len(distractors_feats)}")

    return distractors_id, distractors_feats


def retrieve_sorted(logits, dist_feats, dist_ids):
    score = torch.cosine_similarity(logits.float(), dist_feats.float())
    values, indices = torch.sort(score, descending=True)

    probs = torch.softmax(score / SOFTMAX_TEMPERATURE, dim=0)

    sorted_ids = dist_ids[indices.numpy()]
    sorted_scores = values.detach().cpu().numpy()
    sorted_probs = probs[indices].detach().cpu().numpy()

    return sorted_ids, sorted_scores, sorted_probs


def compute_confidence(sorted_scores, sorted_probs):
    top1_score = float(sorted_scores[0])
    top2_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else np.nan
    gap = float(top1_score - top2_score) if len(sorted_scores) > 1 else np.nan

    conf_softmax = float(sorted_probs[0])
    conf_gap = sigmoid(GAP_SCALE * gap) if not pd.isna(gap) else np.nan

    if CONFIDENCE_METHOD == "softmax":
        conf = conf_softmax
    elif CONFIDENCE_METHOD == "gap_sigmoid":
        conf = conf_gap
    else:
        conf = 0.5 * conf_softmax + 0.5 * conf_gap

    return {
        "confidence": float(conf),
        "confidence_softmax": float(conf_softmax),
        "confidence_gap": float(conf_gap),
        "top1_score": top1_score,
        "top2_score": top2_score,
        "top1_top2_gap": gap,
    }


def run_model_evaluation(run_cfg):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    actual_device = "cuda" if torch.cuda.is_available() else "cpu"
    actual_device_type = "cuda" if actual_device == "cuda" else "cpu"

    ptdtype = {
        "float32": torch.float32,
        "bfloat16": torch.bfloat16,
        "float16": torch.float16,
    }[dtype]

    ctx = (
        nullcontext()
        if actual_device_type == "cpu"
        else torch.amp.autocast(device_type=actual_device_type, dtype=ptdtype)
    )

    print("Device:", actual_device)

    train_dat, test_dat = build_datasets(run_cfg)
    model = build_model(run_cfg, actual_device)

    distractors_id, distractors_feats = build_distractors(
        train_dat,
        test_dat,
        model,
        actual_device,
        ctx,
    )

    rows = []
    k_list = [1, 3, 5, 10, 30, 50]
    count = 0
    skipped = 0

    for outfits, is_correct, set_id in tqdm(test_dat.fitb_questions, desc=f"Evaluate {run_cfg['run_tag']}"):
        correct_indices = np.where(np.array(is_correct))[0]

        if len(correct_indices) == 0:
            skipped += 1
            continue

        correct_idx = int(correct_indices[0])
        target_item_id = outfits[correct_idx]["items"][-1]["item_id"]
        target_item_fg = test_dat.im2fg[target_item_id]

        if target_item_fg not in distractors_feats:
            skipped += 1
            continue

        partial_items = outfits[0]["items"][:-1]
        partial_item_ids = [x.get("item_id") for x in partial_items]

        partial_imgs, partial_txts, _ = test_dat.load_outfit({"items": partial_items})
        partial_imgs, partial_txts, mask = test_dat.pad_imgs_and_txts(
            partial_imgs,
            partial_txts,
        )

        partial_imgs = partial_imgs.to(actual_device)
        partial_txts = partial_txts.to(actual_device)
        mask = mask.to(actual_device)

        target_item_category = torch.tensor(
            test_dat.category_feats[target_item_id]
        ).float().to(actual_device)

        outfit_txt = torch.tensor(
            test_dat.outfit_feats[set_id]
        ).float().to(actual_device)

        with torch.no_grad(), ctx:
            out = model(
                partial_imgs.unsqueeze(0),
                partial_txts.unsqueeze(0),
                mask.unsqueeze(0),
                positive_category=target_item_category.unsqueeze(0),
                outfit_txt=outfit_txt.unsqueeze(0),
            )

            logits = out["logits"].detach().cpu().float()

        cur_dist_id = deepcopy(distractors_id[target_item_fg])
        cur_dist_feats = deepcopy(distractors_feats[target_item_fg])

        if not np.any(cur_dist_id == target_item_id):
            cur_dist_id = np.concatenate([
                cur_dist_id[:-1],
                np.array([target_item_id]),
            ])

            target_emb = test_dat.extract_emb(
                {"item_id": target_item_id},
                model,
                actual_device,
                ctx,
                to_numpy=False,
            ).detach().cpu().float()

            cur_dist_feats = torch.cat([
                cur_dist_feats[:-1, :],
                target_emb,
            ], dim=0)

        sorted_ids, sorted_scores, sorted_probs = retrieve_sorted(
            logits,
            cur_dist_feats,
            cur_dist_id,
        )

        target_pos = np.where(sorted_ids == target_item_id)[0]

        if len(target_pos) == 0:
            skipped += 1
            continue

        rank = int(target_pos[0]) + 1
        target_score = float(sorted_scores[target_pos[0]])

        hits = {k: int(rank <= k) for k in k_list}
        conf_info = compute_confidence(sorted_scores, sorted_probs)

        top10_ids = [to_safe_id(x) for x in sorted_ids[:10]]
        top10_scores = [float(x) for x in sorted_scores[:10]]

        metadata_key = f"{SEED}||{to_safe_id(set_id)}||{to_safe_id(target_item_id)}"

        row = {
            "run_tag": run_cfg["run_tag"],
            "query_key": f"{run_cfg['run_tag']}||{metadata_key}",
            "metadata_key": metadata_key,
            "seed": str(SEED),
            "set_id": to_safe_id(set_id),
            "model_name": run_cfg["model_name"],

            "model_output_top1_id": to_safe_id(sorted_ids[0]),
            "model_output_top10_ids": json.dumps(top10_ids, ensure_ascii=False),
            "confidence": conf_info["confidence"],
            "confidence_band": assign_confidence_band(conf_info["confidence"]),
            "confidence_source": f"{CONFIDENCE_METHOD}: retrieval score proxy",
            "correct_answer": to_safe_id(target_item_id),

            "rank": rank,
            "hit@1": hits[1],
            "hit@3": hits[3],
            "hit@5": hits[5],
            "hit@10": hits[10],
            "hit@30": hits[30],
            "hit@50": hits[50],
            "is_error": int(hits[10] == 0),
            "is_high_conf_error": int(
                conf_info["confidence"] >= HIGH_CONF_THRESHOLD and hits[10] == 0
            ),

            "target_item_id": to_safe_id(target_item_id),
            "target_item_fg": str(target_item_fg),

            "top1_score": conf_info["top1_score"],
            "top2_score": conf_info["top2_score"],
            "target_score": target_score,
            "top1_top2_gap": conf_info["top1_top2_gap"],
            "top10_scores": json.dumps(top10_scores, ensure_ascii=False),

            "partial_item_ids": json.dumps(
                [to_safe_id(x) for x in partial_item_ids],
                ensure_ascii=False,
            ),
        }

        rows.append(row)
        count += 1

        if MAX_EVAL_SAMPLES is not None and count >= MAX_EVAL_SAMPLES:
            break

    print(f"{run_cfg['run_tag']} evaluated:", count)
    print(f"{run_cfg['run_tag']} skipped:", skipped)

    return pd.DataFrame(rows)


# ============================================================
# 6. metadata merge
# ============================================================

def enrich_metadata(df):
    orig_df = load_original_titles(ORIGINAL_TITLE_JSON)
    frag_df = load_fragments(FRAG_JSONL)
    cat_df = load_category_map(CATEGORY_CSV)

    print("Original title rows:", len(orig_df))
    print("Fragment rows:", len(frag_df))
    print("Category rows:", len(cat_df))

    meta = (
        df[["metadata_key", "seed", "set_id", "target_item_id", "target_item_fg"]]
        .drop_duplicates()
        .merge(orig_df, on="set_id", how="left")
        .merge(frag_df, on="set_id", how="left")
        .merge(
            cat_df[["category_id", "fine_category", "major_category"]],
            left_on="target_item_fg",
            right_on="category_id",
            how="left",
        )
    )

    for c in [
        "original_text",
        "title_full_text",
        "weather_text",
        "occasion_text",
        "style_text",
        "fine_category",
        "major_category",
    ]:
        if c not in meta.columns:
            meta[c] = ""
        meta[c] = meta[c].fillna("")

    for c in ["weather_terms", "occasion_terms", "style_terms"]:
        if c not in meta.columns:
            meta[c] = [[] for _ in range(len(meta))]
        meta[c] = meta[c].apply(parse_list_like)

    meta["temperature_c"] = meta.apply(extract_row_temperature, axis=1)

    valid_temp = meta["temperature_c"].dropna()
    temp_threshold = float(valid_temp.median()) if len(valid_temp) > 0 else np.nan

    def assign_weather_group(t):
        if pd.isna(t) or pd.isna(temp_threshold):
            return "unknown"
        return "cold" if t <= temp_threshold else "warm"

    meta["weather_group"] = meta["temperature_c"].apply(assign_weather_group)
    meta["occasion_group"] = meta.apply(assign_occasion_group, axis=1)
    meta["style_count"] = meta["style_terms"].apply(get_style_count)
    meta["style_group"] = np.where(meta["style_count"] >= 2, "high-style", "low-style")
    meta["category_group"] = meta["major_category"].apply(assign_category_group)

    meta_keep_cols = [
        "metadata_key",
        "temperature_c",
        "weather_group",
        "occasion_group",
        "style_count",
        "style_group",
        "fine_category",
        "major_category",
        "category_group",
        "weather_text",
        "occasion_text",
        "style_text",
        "title_full_text",
        "original_text",
    ]

    meta_out = meta[meta_keep_cols].drop_duplicates(subset=["metadata_key"])

    out = df.merge(meta_out, on="metadata_key", how="left")

    for c in ["weather_group", "occasion_group", "style_group", "category_group"]:
        out[c] = out[c].fillna("unknown")

    for c in [
        "fine_category",
        "major_category",
        "weather_text",
        "occasion_text",
        "style_text",
        "title_full_text",
        "original_text",
    ]:
        out[c] = out[c].fillna("")

    return out


# ============================================================
# 7. error type / source / log
# ============================================================

def classify_error(row):
    if row["hit@10"] == 1:
        return "correct"

    if row["is_high_conf_error"] == 1:
        return "high-confidence semantic mismatch"

    if row["category_group"] == "accessory-led":
        return "accessory-led retrieval failure"

    if row["style_group"] == "low-style":
        return "low-style ambiguity"

    if row["weather_group"] == "unknown" or row["occasion_group"] == "unknown":
        return "context metadata incomplete"

    return "general retrieval error"


def infer_possible_source(row):
    error_type = row["error_type"]

    if error_type == "correct":
        return "none"

    if error_type == "high-confidence semantic mismatch":
        return "模型信心高，但正確答案未進入 Top-10；可能代表 confidence calibration 不佳、語意 shortcut，或候選池中存在高度相似干擾項。"

    if error_type == "accessory-led retrieval failure":
        return "配件受天氣與場合約束較弱，模型可能較依賴視覺或風格相似度，導致配件類單品排序錯誤。"

    if error_type == "low-style ambiguity":
        return "輸入描述中的 style fragment 較少，語意條件不足，模型可能用少量 token 做過度推論。"

    if error_type == "context metadata incomplete":
        return "天氣或場合資訊未被解析，導致模型失敗來源較難診斷。"

    return "候選單品相似度高，或模型未充分捕捉 partial outfit 與 target item 的細節相容性。"


def infer_risk_level(row):
    if row["hit@10"] == 1:
        return "low"

    if row["is_high_conf_error"] == 1:
        if row["occasion_group"] == "formal":
            return "medium-high"
        return "medium"

    return "low-medium"


def make_observable_log(row):
    log = {
        "run_tag": str(row.get("run_tag", "")),
        "model_name": str(row.get("model_name", "")),
        "query_key": str(row.get("query_key", "")),
        "metadata_key": str(row.get("metadata_key", "")),
        "set_id": str(row.get("set_id", "")),
        "target_item_id": str(row.get("target_item_id", "")),
        "correct_answer": str(row.get("correct_answer", "")),
        "model_output_top1_id": str(row.get("model_output_top1_id", "")),
        "model_output_top10_ids": parse_list_like(row.get("model_output_top10_ids", "[]")),
        "rank": to_float(row.get("rank", np.nan)),
        "hit_at_10": int(row.get("hit@10", 0)),
        "confidence": to_float(row.get("confidence", np.nan)),
        "confidence_band": str(row.get("confidence_band", "")),
        "top1_score": to_float(row.get("top1_score", np.nan)),
        "top2_score": to_float(row.get("top2_score", np.nan)),
        "target_score": to_float(row.get("target_score", np.nan)),
        "top1_top2_gap": to_float(row.get("top1_top2_gap", np.nan)),
        "target_item_fg": str(row.get("target_item_fg", "")),
        "fine_category": str(row.get("fine_category", "")),
        "major_category": str(row.get("major_category", "")),
        "temperature_c": to_float(row.get("temperature_c", np.nan)),
        "weather_group": str(row.get("weather_group", "")),
        "occasion_group": str(row.get("occasion_group", "")),
        "style_count": to_float(row.get("style_count", np.nan)),
        "style_group": str(row.get("style_group", "")),
        "category_group": str(row.get("category_group", "")),
        "weather_text": str(row.get("weather_text", "")),
        "occasion_text": str(row.get("occasion_text", "")),
        "style_text": str(row.get("style_text", "")),
        "error_type": str(row.get("error_type", "")),
        "possible_source": str(row.get("possible_source", "")),
        "risk_level": str(row.get("risk_level", "")),
    }

    return json.dumps(log, ensure_ascii=False)


# ============================================================
# 8. metrics and plotting helpers
# ============================================================

def compute_ece(data, n_bins=10):
    temp = data[["confidence", "hit@10"]].dropna().copy()
    temp["confidence"] = temp["confidence"].clip(0, 1)

    bins = np.linspace(0, 1, n_bins + 1)
    temp["bin"] = pd.cut(temp["confidence"], bins=bins, include_lowest=True)

    rows = []
    ece = 0.0

    for b, g in temp.groupby("bin", observed=False):
        if len(g) == 0:
            continue

        avg_conf = g["confidence"].mean()
        acc = g["hit@10"].mean()
        weight = len(g) / len(temp)
        gap = abs(avg_conf - acc)

        ece += weight * gap

        rows.append({
            "bin": str(b),
            "n": len(g),
            "avg_confidence": avg_conf,
            "hit10_accuracy": acc,
            "gap": gap,
        })

    return float(ece), pd.DataFrame(rows)


def brier_score(data):
    temp = data[["confidence", "hit@10"]].dropna().copy()
    return float(np.mean((temp["confidence"].clip(0, 1) - temp["hit@10"]) ** 2))


def save_df_as_image(data, title, out_path, max_rows=12, font_size=8, wrap_width=18):
    plot_df = data.head(max_rows).copy()

    for col in plot_df.columns:
        if pd.api.types.is_float_dtype(plot_df[col]):
            plot_df[col] = plot_df[col].map(lambda x: f"{x:.3f}" if pd.notna(x) else "NA")
        else:
            plot_df[col] = plot_df[col].map(lambda x: wrap_text(x, width=wrap_width))

    fig_height = max(2.8, 0.55 * len(plot_df) + 1.4)
    fig_width = max(10, 1.35 * len(plot_df.columns))

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")
    ax.set_title(title, fontsize=15, fontweight="bold", pad=14)

    table = ax.table(
        cellText=plot_df.values,
        colLabels=plot_df.columns,
        loc="center",
        cellLoc="left",
        colLoc="left",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1.5)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_height(cell.get_height() * 1.15)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def summarize_one_run(g):
    ece, _ = compute_ece(g, N_BINS)

    return {
        "n": len(g),
        "Hit@1": g["hit@1"].mean(),
        "Hit@10": g["hit@10"].mean(),
        "Median Rank": g["rank"].median(),
        "Mean Rank": g["rank"].mean(),
        "Avg Confidence": g["confidence"].mean(),
        "Error Rate": g["is_error"].mean(),
        "High-conf Error Rate": g["is_high_conf_error"].mean(),
        "ECE": ece,
        "Brier Score": brier_score(g),
    }


def make_comparison_plots(required_fields):
    # --------------------------------------------------------
    # plot_01: performance comparison table
    # --------------------------------------------------------
    summary_rows = []

    for run_tag, g in required_fields.groupby("run_tag"):
        row = summarize_one_run(g)
        row["run_tag"] = run_tag
        summary_rows.append(row)

    overall_df = pd.DataFrame(summary_rows)

    if BASELINE_TAG in overall_df["run_tag"].values and PROPOSED_TAG in overall_df["run_tag"].values:
        base = overall_df[overall_df["run_tag"] == BASELINE_TAG].iloc[0]
        prop = overall_df[overall_df["run_tag"] == PROPOSED_TAG].iloc[0]

        delta_row = {
            "run_tag": "Δ proposed - original",
            "n": "",
            "Hit@1": prop["Hit@1"] - base["Hit@1"],
            "Hit@10": prop["Hit@10"] - base["Hit@10"],
            "Median Rank": prop["Median Rank"] - base["Median Rank"],
            "Mean Rank": prop["Mean Rank"] - base["Mean Rank"],
            "Avg Confidence": prop["Avg Confidence"] - base["Avg Confidence"],
            "Error Rate": prop["Error Rate"] - base["Error Rate"],
            "High-conf Error Rate": prop["High-conf Error Rate"] - base["High-conf Error Rate"],
            "ECE": prop["ECE"] - base["ECE"],
            "Brier Score": prop["Brier Score"] - base["Brier Score"],
        }

        overall_df = pd.concat([overall_df, pd.DataFrame([delta_row])], ignore_index=True)

    perf_img = overall_df.copy()

    for col in ["Hit@1", "Hit@10", "Error Rate", "High-conf Error Rate"]:
        if col in perf_img.columns:
            perf_img[col] = perf_img[col].map(
                lambda x: pct(x) if isinstance(x, (int, float, np.floating)) else x
            )

    for col in ["Median Rank", "Mean Rank", "Avg Confidence", "ECE", "Brier Score"]:
        if col in perf_img.columns:
            perf_img[col] = perf_img[col].map(
                lambda x: f"{x:.3f}" if isinstance(x, (int, float, np.floating)) else x
            )

    save_df_as_image(
        perf_img,
        "Performance Comparison: Original vs Proposed",
        os.path.join(OUTPUT_DIR, "plot_01_performance_comparison.png"),
        max_rows=5,
        font_size=8,
        wrap_width=16,
    )

    # --------------------------------------------------------
    # plot_02: confidence distribution comparison
    # --------------------------------------------------------
    plt.figure(figsize=(9, 4.8))

    bins = np.linspace(0.5, 1.0, 26)

    for run_tag, g in required_fields.groupby("run_tag"):
        plt.hist(
            g["confidence"].dropna(),
            bins=bins,
            histtype="step",
            linewidth=2,
            label=f"{run_tag} mean={g['confidence'].mean():.3f}",
        )

    plt.axvline(
        HIGH_CONF_THRESHOLD,
        linestyle=":",
        linewidth=2,
        label=f"High-conf threshold = {HIGH_CONF_THRESHOLD}",
    )

    plt.xlabel("Confidence")
    plt.ylabel("Number of samples")
    plt.title("Confidence Distribution: Original vs Proposed")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plot_02_confidence_distribution_comparison.png"), dpi=300)
    plt.close()

    # --------------------------------------------------------
    # plot_03: reliability diagram comparison
    # --------------------------------------------------------
    plt.figure(figsize=(6, 6))

    for run_tag, g in required_fields.groupby("run_tag"):
        ece, calib = compute_ece(g, N_BINS)

        if len(calib) > 0:
            plt.plot(
                calib["avg_confidence"],
                calib["hit10_accuracy"],
                marker="o",
                linewidth=2,
                label=f"{run_tag}, ECE={ece:.3f}",
            )

    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=2, label="Perfect calibration")
    plt.xlabel("Average confidence")
    plt.ylabel("Empirical Hit@10 accuracy")
    plt.title("Reliability Diagram: Original vs Proposed")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plot_03_reliability_diagram_comparison.png"), dpi=300)
    plt.close()

    # --------------------------------------------------------
    # plot_04: confidence band comparison table
    # --------------------------------------------------------
    band = (
        required_fields.groupby(["confidence_band", "run_tag"], dropna=False)
        .agg(
            n=("set_id", "count"),
            hit10=("hit@10", "mean"),
            error_rate=("is_error", "mean"),
            high_conf_error_rate=("is_high_conf_error", "mean"),
            median_rank=("rank", "median"),
        )
        .reset_index()
    )

    rows = []

    for band_name, g in band.groupby("confidence_band"):
        row = {"confidence_band": band_name}

        for run_tag in [BASELINE_TAG, PROPOSED_TAG]:
            gg = g[g["run_tag"] == run_tag]

            if len(gg) == 0:
                row[f"{run_tag}_n"] = ""
                row[f"{run_tag}_hit10"] = ""
                row[f"{run_tag}_hce"] = ""
                row[f"{run_tag}_median_rank"] = ""
            else:
                r = gg.iloc[0]
                row[f"{run_tag}_n"] = int(r["n"])
                row[f"{run_tag}_hit10"] = pct(r["hit10"])
                row[f"{run_tag}_hce"] = pct(r["high_conf_error_rate"])
                row[f"{run_tag}_median_rank"] = f"{r['median_rank']:.1f}"

        rows.append(row)

    band_img = pd.DataFrame(rows)

    save_df_as_image(
        band_img,
        "Confidence Band Summary: Original vs Proposed",
        os.path.join(OUTPUT_DIR, "plot_04_confidence_band_comparison.png"),
        max_rows=10,
        font_size=8,
        wrap_width=14,
    )

    # --------------------------------------------------------
    # plot_05: error slicing comparison
    # --------------------------------------------------------
    error_slicing = (
        required_fields[required_fields["is_error"] == 1]
        .groupby(["error_type", "run_tag"], dropna=False)
        .agg(n=("set_id", "count"))
        .reset_index()
    )

    pivot_error = (
        error_slicing
        .pivot(index="error_type", columns="run_tag", values="n")
        .fillna(0)
    )

    ax = pivot_error.plot(kind="bar", figsize=(10, 5))
    ax.set_ylabel("Number of errors")
    ax.set_title("Error Slicing by Error Type: Original vs Proposed")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plot_05_error_slicing_comparison.png"), dpi=300)
    plt.close()

    # --------------------------------------------------------
    # subgroup metrics table
    # --------------------------------------------------------
    subgroup_parts = []

    for run_tag, run_df in required_fields.groupby("run_tag"):
        for col in ["weather_group", "occasion_group", "style_group", "category_group"]:
            temp = (
                run_df.groupby(col, dropna=False)
                .agg(
                    n=("set_id", "count"),
                    hit10=("hit@10", "mean"),
                    avg_confidence=("confidence", "mean"),
                    high_conf_error_rate=("is_high_conf_error", "mean"),
                    median_rank=("rank", "median"),
                )
                .reset_index()
            )

            temp["run_tag"] = run_tag
            temp["subgroup_type"] = col
            temp["subgroup"] = temp[col].fillna("unknown").astype(str)

            if (temp["subgroup"] != "unknown").any():
                temp = temp[temp["subgroup"] != "unknown"]

            subgroup_parts.append(temp[[
                "run_tag",
                "subgroup_type",
                "subgroup",
                "n",
                "hit10",
                "avg_confidence",
                "high_conf_error_rate",
                "median_rank",
            ]])

    subgroup_metrics = pd.concat(subgroup_parts, ignore_index=True)
    subgroup_metrics["label"] = subgroup_metrics["subgroup_type"] + "\n" + subgroup_metrics["subgroup"]

    # --------------------------------------------------------
    # plot_06: subgroup hit10 comparison
    # --------------------------------------------------------
    pivot_hit = (
        subgroup_metrics
        .pivot_table(index="label", columns="run_tag", values="hit10")
        .fillna(0)
    )

    ax = pivot_hit.plot(kind="bar", figsize=(12, 5.5))
    ax.set_ylabel("Hit@10")
    ax.set_title("Subgroup Metrics: Hit@10, Original vs Proposed")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plot_06_subgroup_hit10_comparison.png"), dpi=300)
    plt.close()

    # --------------------------------------------------------
    # plot_07: subgroup high-confidence error comparison
    # --------------------------------------------------------
    pivot_hce = (
        subgroup_metrics
        .pivot_table(index="label", columns="run_tag", values="high_conf_error_rate")
        .fillna(0)
    )

    ax = pivot_hce.plot(kind="bar", figsize=(12, 5.5))
    ax.set_ylabel("High-confidence error rate")
    ax.set_title("Subgroup Metrics: High-confidence Error Rate, Original vs Proposed")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "plot_07_subgroup_high_conf_error_comparison.png"), dpi=300)
    plt.close()

    # --------------------------------------------------------
    # plot_08: subgroup delta hit10 heatmap
    # --------------------------------------------------------
    if BASELINE_TAG in subgroup_metrics["run_tag"].values and PROPOSED_TAG in subgroup_metrics["run_tag"].values:
        pivot_delta = (
            subgroup_metrics
            .pivot_table(
                index=["subgroup_type", "subgroup"],
                columns="run_tag",
                values="hit10",
            )
            .dropna()
        )

        pivot_delta["delta_hit10"] = pivot_delta[PROPOSED_TAG] - pivot_delta[BASELINE_TAG]
        pivot_delta = pivot_delta.reset_index()

        row_order = ["weather_group", "occasion_group", "style_group", "category_group"]
        col_order = list(pivot_delta["subgroup"].unique())

        matrix = pd.DataFrame(index=row_order, columns=col_order, dtype=float)

        for _, r in pivot_delta.iterrows():
            matrix.loc[r["subgroup_type"], r["subgroup"]] = r["delta_hit10"]

        matrix = matrix.dropna(how="all")
        matrix = matrix.loc[:, matrix.notna().any(axis=0)]

        fig, ax = plt.subplots(figsize=(max(7, len(matrix.columns) * 1.3), max(3.5, len(matrix.index) * 0.9 + 1.5)))
        im = ax.imshow(matrix.values.astype(float), aspect="auto")

        ax.set_xticks(range(len(matrix.columns)))
        ax.set_yticks(range(len(matrix.index)))
        ax.set_xticklabels(matrix.columns, rotation=30, ha="right")
        ax.set_yticklabels(matrix.index)

        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                val = matrix.iloc[i, j]
                if not pd.isna(val):
                    ax.text(j, i, f"{val:+.3f}", ha="center", va="center", fontsize=9, fontweight="bold")

        plt.colorbar(im, ax=ax, label="ΔHit@10 = proposed - original")
        ax.set_title("Subgroup ΔHit@10 Heatmap: Proposed - Original", fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "plot_08_subgroup_delta_hit10_heatmap.png"), dpi=300)
        plt.close()

    # --------------------------------------------------------
    # plot_09: proposed high-confidence error examples with original comparison
    # --------------------------------------------------------
    proposed = required_fields[required_fields["run_tag"] == PROPOSED_TAG].copy()
    original = required_fields[required_fields["run_tag"] == BASELINE_TAG].copy()

    original_small = original[[
        "metadata_key",
        "rank",
        "hit@10",
        "confidence",
        "model_output_top1_id",
    ]].rename(columns={
        "rank": "original_rank",
        "hit@10": "original_hit10",
        "confidence": "original_confidence",
        "model_output_top1_id": "original_top1_id",
    })

    hce = (
        proposed[proposed["is_high_conf_error"] == 1]
        .merge(original_small, on="metadata_key", how="left")
        .sort_values(["confidence", "rank"], ascending=[False, False])
        .copy()
    )

    hce["rank_change_vs_original"] = hce["rank"] - hce["original_rank"]

    hce_img_cols = [
        "set_id",
        "target_item_id",
        "correct_answer",
        "model_output_top1_id",
        "original_top1_id",
        "rank",
        "original_rank",
        "rank_change_vs_original",
        "confidence",
        "original_confidence",
        "fine_category",
        "major_category",
        "weather_group",
        "occasion_group",
        "style_group",
        "category_group",
        "error_type",
    ]

    hce_img_cols = [c for c in hce_img_cols if c in hce.columns]

    save_df_as_image(
        hce[hce_img_cols].head(3),
        "Proposed High-confidence Errors with Original Comparison",
        os.path.join(OUTPUT_DIR, "plot_09_high_confidence_errors_comparison.png"),
        max_rows=3,
        font_size=7,
        wrap_width=14,
    )

    return overall_df


# ============================================================
# 9. main
# ============================================================

def main():
    all_runs = []

    for run_cfg in RUN_CONFIGS:
        print("\n" + "=" * 80)
        print("Running evaluation:", run_cfg["run_tag"])
        print("=" * 80)

        one_run = run_model_evaluation(run_cfg)
        all_runs.append(one_run)

    required_fields = pd.concat(all_runs, ignore_index=True)

    required_fields = enrich_metadata(required_fields)

    required_fields["error_type"] = required_fields.apply(classify_error, axis=1)
    required_fields["possible_source"] = required_fields.apply(infer_possible_source, axis=1)
    required_fields["risk_level"] = required_fields.apply(infer_risk_level, axis=1)
    required_fields["observable_log"] = required_fields.apply(make_observable_log, axis=1)

    needed_cols = [
        "run_tag",
        "query_key",
        "metadata_key",
        "seed",
        "set_id",
        "model_name",

        "model_output_top1_id",
        "model_output_top10_ids",
        "confidence",
        "confidence_band",
        "confidence_source",
        "correct_answer",

        "rank",
        "hit@1",
        "hit@3",
        "hit@5",
        "hit@10",
        "hit@30",
        "hit@50",
        "is_error",
        "is_high_conf_error",

        "error_type",
        "possible_source",
        "observable_log",
        "risk_level",

        "target_item_id",
        "target_item_fg",
        "fine_category",
        "major_category",
        "temperature_c",
        "weather_group",
        "occasion_group",
        "style_count",
        "style_group",
        "category_group",

        "top1_score",
        "top2_score",
        "target_score",
        "top1_top2_gap",

        "weather_text",
        "occasion_text",
        "style_text",
        "title_full_text",
        "original_text",
    ]

    for col in needed_cols:
        if col not in required_fields.columns:
            required_fields[col] = ""

    required_fields = required_fields[needed_cols].copy()

    out_csv = os.path.join(OUTPUT_DIR, "reliability_required_fields.csv")
    required_fields.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print("Saved required fields:", out_csv)

    overall_df = make_comparison_plots(required_fields)

    print("\n完成。只保留必要 CSV + 兩模型比較圖。")
    print("CSV:", out_csv)
    print("\nPerformance comparison:")
    print(overall_df)
    print("\n下一步請執行：python 02_reliability_dashboard.py")


if __name__ == "__main__":
    main()

中文字型設定完成： Microsoft JhengHei
使用字型路徑： /mnt/c/Windows/Fonts/msjh.ttc

Running evaluation: original
Device: cuda
Checkpoint loaded for original: ./experiments/cir_old_seed1/ckpt.pt


Fill distractors from train: 100%|████████████████████████████████████████████████████| 149/149 [00:13<00:00, 10.88it/s]


Selected fine-grained categories: 19


Evaluate original: 100%|█████████████████████████████████████████████████████████| 15145/15145 [00:31<00:00, 487.09it/s]


original evaluated: 9311
original skipped: 5834

Running evaluation: proposed
Device: cuda
Checkpoint loaded for proposed: ./experiments/cir_new_seed1/ckpt.pt


Fill distractors from train: 100%|████████████████████████████████████████████████████| 149/149 [00:15<00:00,  9.79it/s]


Selected fine-grained categories: 19


Evaluate proposed: 100%|█████████████████████████████████████████████████████████| 15145/15145 [00:30<00:00, 501.45it/s]


proposed evaluated: 9311
proposed skipped: 5834
Original title rows: 68306
Fragment rows: 35140
Category rows: 159
Saved required fields: ./reliability_outputs/compare/reliability_required_fields.csv

完成。只保留必要 CSV + 兩模型比較圖。
CSV: ./reliability_outputs/compare/reliability_required_fields.csv

Performance comparison:
      n     Hit@1    Hit@10  Median Rank   Mean Rank  Avg Confidence  \
0  9311  0.013855  0.077972        230.0  463.324777        0.720271   
1  9311  0.015251  0.087101        205.0  441.759102        0.718550   
2        0.001396  0.009129        -25.0  -21.565675       -0.001721   

   Error Rate  High-conf Error Rate       ECE  Brier Score  \
0    0.922028              0.219740  0.642299     0.504966   
1    0.912899              0.206315  0.631449     0.497345   
2   -0.009129             -0.013425 -0.010850    -0.007621   

                 run_tag  
0               original  
1               proposed  
2  Δ proposed - original  

下一步請執行：python 02_reliability_dashboar

In [22]:
# ============================================================
# 02_reliability_dashboard.py
#
# 功能：
# 1. 重新整理 dashboard 版面：圖片集中成圖卡
# 2. 點選圖片才顯示詳細說明
# 3. 首頁顯示「如何查看資料」引導
# 4. 自動整理：
#    - performance
#    - confidence distribution
#    - error slicing
#    - subgroup metrics
#    - 3 個 high-confidence error 分析
#    - 可能錯誤來源與修正策略
#    - real-world risk assessment
# 5. 一個按鍵輸出報告書 reliability_report.html
#
# 輸入：
# - reliability_required_fields.csv
# - plot_*.png
#
# 輸出：
# - reliability_dashboard.html
# - reliability_report.html
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from html import escape


# ============================================================
# 1. 路徑設定
# ============================================================

OUTPUT_DIR = "./reliability_outputs/compare"

REQUIRED_FIELDS_CSV = os.path.join(OUTPUT_DIR, "reliability_required_fields.csv")
DASHBOARD_HTML = os.path.join(OUTPUT_DIR, "reliability_dashboard.html")
REPORT_HTML = os.path.join(OUTPUT_DIR, "reliability_report.html")

BASELINE_TAG = "original"
PROPOSED_TAG = "proposed"
HIGH_CONF_THRESHOLD = 0.85

if not os.path.exists(REQUIRED_FIELDS_CSV):
    raise FileNotFoundError(f"找不到必要 CSV：{REQUIRED_FIELDS_CSV}")


# ============================================================
# 2. 讀取資料
# ============================================================

df = pd.read_csv(REQUIRED_FIELDS_CSV)

numeric_cols = [
    "confidence",
    "rank",
    "hit@1",
    "hit@3",
    "hit@5",
    "hit@10",
    "hit@30",
    "hit@50",
    "is_error",
    "is_high_conf_error",
    "top1_score",
    "top2_score",
    "target_score",
    "top1_top2_gap",
    "temperature_c",
    "style_count",
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["hit@10"] = df["hit@10"].fillna(0).astype(int)
df["is_error"] = (df["hit@10"] == 0).astype(int)
df["is_high_conf_error"] = (
    (df["confidence"] >= HIGH_CONF_THRESHOLD) & (df["is_error"] == 1)
).astype(int)

if "metadata_key" not in df.columns:
    df["metadata_key"] = (
        df["seed"].astype(str)
        + "||"
        + df["set_id"].astype(str)
        + "||"
        + df["target_item_id"].astype(str)
    )


# ============================================================
# 3. 指標函式
# ============================================================

def pct(x):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:.2f}%"

def pp(x):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:+.2f} pp"

def fmt(x, digits=3):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

def compute_ece(data, n_bins=10):
    temp = data[["confidence", "hit@10"]].dropna().copy()

    if len(temp) == 0:
        return np.nan

    temp["confidence"] = temp["confidence"].clip(0, 1)
    bins = np.linspace(0, 1, n_bins + 1)
    temp["bin"] = pd.cut(temp["confidence"], bins=bins, include_lowest=True)

    ece = 0.0

    for _, g in temp.groupby("bin", observed=False):
        if len(g) == 0:
            continue

        avg_conf = g["confidence"].mean()
        acc = g["hit@10"].mean()
        weight = len(g) / len(temp)
        ece += weight * abs(avg_conf - acc)

    return float(ece)

def brier_score(data):
    temp = data[["confidence", "hit@10"]].dropna().copy()

    if len(temp) == 0:
        return np.nan

    return float(np.mean((temp["confidence"].clip(0, 1) - temp["hit@10"]) ** 2))

def summarize_one_run(g):
    return {
        "n": len(g),
        "hit1": g["hit@1"].mean(),
        "hit10": g["hit@10"].mean(),
        "median_rank": g["rank"].median(),
        "mean_rank": g["rank"].mean(),
        "avg_confidence": g["confidence"].mean(),
        "error_rate": g["is_error"].mean(),
        "high_conf_error_rate": g["is_high_conf_error"].mean(),
        "ece": compute_ece(g),
        "brier": brier_score(g),
    }

summary = {}

for run_tag, g in df.groupby("run_tag"):
    summary[run_tag] = summarize_one_run(g)

base = summary.get(BASELINE_TAG)
prop = summary.get(PROPOSED_TAG)

if base is None or prop is None:
    raise ValueError("CSV 中需要同時包含 run_tag = original 與 run_tag = proposed。")


# ============================================================
# 4. 圖片檔案自動偵測
# ============================================================

def first_existing(candidates):
    for fname in candidates:
        if os.path.exists(os.path.join(OUTPUT_DIR, fname)):
            return fname
    return candidates[0]

IMAGES = {
    "performance": first_existing([
        "plot_01_performance_comparison.png",
        "plot_01_performance_summary.png",
    ]),
    "confidence": first_existing([
        "plot_02_confidence_distribution_comparison.png",
        "plot_02_confidence_distribution.png",
    ]),
    "reliability": first_existing([
        "plot_03_reliability_diagram_comparison.png",
        "plot_03_reliability_diagram.png",
    ]),
    "confidence_band": first_existing([
        "plot_04_confidence_band_comparison.png",
        "plot_07_confidence_band_table.png",
    ]),
    "error_slicing": first_existing([
        "plot_05_error_slicing_comparison.png",
        "plot_04_error_slicing.png",
    ]),
    "subgroup_hit10": first_existing([
        "plot_06_subgroup_hit10_comparison.png",
        "plot_05_subgroup_hit10.png",
    ]),
    "subgroup_hce": first_existing([
        "plot_07_subgroup_high_conf_error_comparison.png",
        "plot_06_subgroup_high_conf_error.png",
    ]),
    "subgroup_delta": first_existing([
        "plot_08_subgroup_delta_hit10_heatmap.png",
        "plot_10_subset_delta_heatmap.png",
    ]),
    "hce_cases": first_existing([
        "plot_09_high_confidence_errors_comparison.png",
        "plot_08_high_confidence_errors.png",
    ]),
}


# ============================================================
# 5. High-confidence error cases
# ============================================================

def safe_value(row, col, default=""):
    if col not in row:
        return default
    v = row[col]
    if pd.isna(v):
        return default
    return v

prop_df = df[df["run_tag"] == PROPOSED_TAG].copy()
base_df = df[df["run_tag"] == BASELINE_TAG].copy()

base_compare_cols = [
    "metadata_key",
    "model_output_top1_id",
    "rank",
    "confidence",
    "hit@10",
]

base_compare = base_df[base_compare_cols].rename(columns={
    "model_output_top1_id": "original_top1_id",
    "rank": "original_rank",
    "confidence": "original_confidence",
    "hit@10": "original_hit10",
})

hce = (
    prop_df[prop_df["is_high_conf_error"] == 1]
    .sort_values(["confidence", "rank"], ascending=[False, False])
    .head(3)
    .merge(base_compare, on="metadata_key", how="left")
)

def build_case_analysis(row, idx):
    rank = safe_value(row, "rank", np.nan)
    original_rank = safe_value(row, "original_rank", np.nan)
    confidence = safe_value(row, "confidence", np.nan)
    original_conf = safe_value(row, "original_confidence", np.nan)

    if pd.notna(original_rank):
        rank_change = rank - original_rank

        if rank_change < 0:
            compare_sentence = (
                f"相較 Original，Proposed 的正確答案排名前進了 {abs(rank_change):.0f} 名，"
                f"但 rank 仍為 {rank:.0f}，沒有進入 Top-10。"
            )
        elif rank_change > 0:
            compare_sentence = (
                f"相較 Original，Proposed 的正確答案排名退後了 {rank_change:.0f} 名，"
                f"表示這個案例中 Proposed 雖然信心高，但排序反而更差。"
            )
        else:
            compare_sentence = (
                "Proposed 與 Original 的正確答案排名相同，但仍沒有進入 Top-10。"
            )
    else:
        compare_sentence = "此案例沒有找到可對應的 Original 結果，因此主要分析 Proposed 本身的錯誤。"

    category = safe_value(row, "fine_category", "unknown")
    major = safe_value(row, "major_category", "unknown")
    weather = safe_value(row, "weather_group", "unknown")
    occasion = safe_value(row, "occasion_group", "unknown")
    style = safe_value(row, "style_group", "unknown")
    error_type = safe_value(row, "error_type", "unknown")
    source = safe_value(row, "possible_source", "")

    return {
        "title": f"High-confidence Error Case {idx}",
        "summary": (
            f"Proposed confidence = {fmt(confidence, 3)}，"
            f"rank = {fmt(rank, 0)}，"
            f"target = {escape(str(safe_value(row, 'target_item_id')))}，"
            f"top1 = {escape(str(safe_value(row, 'model_output_top1_id')))}。"
        ),
        "analysis": f"""
        這是一個 high-confidence error：Proposed 模型給出非常高的 confidence，
        但正確答案沒有進入 Top-10。{compare_sentence}
        這表示 Proposed 即使在 performance 上相較 Original 有改善，仍可能在個別案例中非常有信心地排錯。

        此案例的 target item 類別為 <b>{escape(str(category))}</b>，
        major category 為 <b>{escape(str(major))}</b>；
        subgroup 為 weather = <b>{escape(str(weather))}</b>、
        occasion = <b>{escape(str(occasion))}</b>、
        style = <b>{escape(str(style))}</b>。
        錯誤類型被歸為 <b>{escape(str(error_type))}</b>。

        可能來源：{escape(str(source))}
        """,
        "log": {
            "set_id": safe_value(row, "set_id"),
            "correct_answer": safe_value(row, "correct_answer"),
            "proposed_top1": safe_value(row, "model_output_top1_id"),
            "original_top1": safe_value(row, "original_top1_id"),
            "proposed_rank": safe_value(row, "rank"),
            "original_rank": safe_value(row, "original_rank"),
            "proposed_confidence": safe_value(row, "confidence"),
            "original_confidence": safe_value(row, "original_confidence"),
            "error_type": safe_value(row, "error_type"),
            "risk_level": safe_value(row, "risk_level"),
        }
    }

case_cards = [
    build_case_analysis(row, i + 1)
    for i, (_, row) in enumerate(hce.iterrows())
]


# ============================================================
# 6. 圖片卡說明
# ============================================================

metric_delta_hit10 = prop["hit10"] - base["hit10"]
metric_delta_rank = prop["median_rank"] - base["median_rank"]
metric_delta_ece = prop["ece"] - base["ece"]
metric_delta_hce = prop["high_conf_error_rate"] - base["high_conf_error_rate"]

image_cards = [
    {
        "id": "performance",
        "title": "Performance Comparison",
        "subtitle": "先看 Proposed 是否比 Original 更準。",
        "img": IMAGES["performance"],
        "required": "performance",
        "takeaway": (
            f"Proposed Hit@10 = {pct(prop['hit10'])}，Original = {pct(base['hit10'])}，"
            f"差異 = {pp(metric_delta_hit10)}。Median Rank 差異 = {metric_delta_rank:+.0f}。"
        ),
        "detail": """
        這張圖是整份 dashboard 的起點。它比較 Original baseline 與 Proposed model 的整體表現。
        閱讀時先看 Hit@10 是否上升，再看 Median Rank 是否下降。
        Hit@10 上升代表正確答案更常進入前 10 名；
        Median Rank 下降代表正確答案整體被排得更前面。

        這裡也要同時看 ECE 與 High-confidence Error Rate。
        如果 Proposed 的 Hit@10 變好，但 ECE 仍高，代表它雖然更準，
        但 confidence 仍不一定能解讀成「真的正確的機率」。
        """
    },
    {
        "id": "confidence",
        "title": "Confidence Distribution",
        "subtitle": "比較兩模型信心分數分布。",
        "img": IMAGES["confidence"],
        "required": "confidence distribution",
        "takeaway": (
            f"Original 平均 confidence = {fmt(base['avg_confidence'])}，"
            f"Proposed = {fmt(prop['avg_confidence'])}。"
        ),
        "detail": """
        這張圖用來看兩個模型是否常輸出高 confidence。
        不過 confidence distribution 本身不能代表模型可靠。
        一個模型可以經常輸出高 confidence，但實際上仍然常錯。

        因此，這張圖要和 Reliability Diagram 以及 Confidence Band Summary 一起看。
        如果 Proposed 的 confidence 分布與 Original 類似，但 Hit@10 提升，
        代表 performance 改善不是單純來自更高的 confidence。
        """
    },
    {
        "id": "reliability",
        "title": "Reliability Diagram",
        "subtitle": "檢查 confidence 是否校準。",
        "img": IMAGES["reliability"],
        "required": "confidence calibration",
        "takeaway": (
            f"Original ECE = {fmt(base['ece'])}，Proposed ECE = {fmt(prop['ece'])}，"
            f"差異 = {metric_delta_ece:+.3f}。"
        ),
        "detail": """
        Reliability Diagram 用來比較 confidence 與實際 Hit@10 accuracy 是否一致。
        理想情況下，模型曲線應接近對角線。
        若曲線明顯低於對角線，代表模型 over-confident：
        它給出的 confidence 高於實際正確率。

        如果 Proposed 的 ECE 比 Original 低，代表 calibration 有些改善；
        但只要曲線仍然遠低於對角線，就表示 Proposed 仍需要 confidence calibration。
        """
    },
    {
        "id": "confidence_band",
        "title": "Confidence Band Summary",
        "subtitle": "看高 confidence 區間是否真的比較準。",
        "img": IMAGES["confidence_band"],
        "required": "confidence distribution",
        "takeaway": "重點看 0.90–1.00 區間：高信心是否仍然高錯誤率。",
        "detail": """
        這張表把樣本依 confidence 區間分組，直接檢查高信心區間是否對應較高 Hit@10。
        如果 0.90–1.00 區間的 Hit@10 仍偏低，
        或 high-confidence error rate 很高，就代表模型在高信心下仍可能錯。

        這張圖是挑選 high-confidence error 進行個案分析的主要依據。
        """
    },
    {
        "id": "error_slicing",
        "title": "Error Slicing",
        "subtitle": "比較錯誤集中在哪些類型。",
        "img": IMAGES["error_slicing"],
        "required": "error slicing",
        "takeaway": "觀察 Proposed 的錯誤是否比 Original 少，以及哪些錯誤仍最多。",
        "detail": """
        Error slicing 比較 Original 與 Proposed 的錯誤類型分布。
        如果 Proposed 的總錯誤數下降，代表 performance 有改善；
        但如果 high-confidence semantic mismatch 仍然很多，
        代表 reliability failure 仍然存在。

        特別是 accessory-led retrieval failure 若仍是主要錯誤來源，
        表示模型在配件類 target item 上仍比較弱。
        """
    },
    {
        "id": "subgroup_hit10",
        "title": "Subgroup Metrics: Hit@10",
        "subtitle": "看不同子群體是否都有改善。",
        "img": IMAGES["subgroup_hit10"],
        "required": "subgroup metrics",
        "takeaway": "比較 cold/warm、formal/casual、style、category 各群體的 Hit@10。",
        "detail": """
        這張圖比較兩個模型在不同 subgroup 上的 Hit@10。
        如果 Proposed 在多數 subgroup 都高於 Original，
        代表改善不是只來自單一子群體，而是較全面的提升。

        如果某個 subgroup 仍然偏低，例如 accessory-led，
        代表該類別仍是模型弱點，需要後續針對性修正。
        """
    },
    {
        "id": "subgroup_hce",
        "title": "Subgroup Metrics: High-confidence Error Rate",
        "subtitle": "看哪個子群體最容易「很有信心但錯」。",
        "img": IMAGES["subgroup_hce"],
        "required": "subgroup metrics",
        "takeaway": (
            f"Proposed high-confidence error rate = {pct(prop['high_conf_error_rate'])}，"
            f"Original = {pct(base['high_conf_error_rate'])}。"
        ),
        "detail": """
        這張圖比 Hit@10 更接近 reliability 的角度，
        因為它直接觀察「模型很有信心卻仍然錯」的比例。

        如果 Proposed 在某些 subgroup 的 high-confidence error rate 沒有下降，
        代表該群體雖然可能 performance 有改善，但使用者仍可能過度相信錯誤推薦。
        """
    },
    {
        "id": "subgroup_delta",
        "title": "Subgroup ΔHit@10 Heatmap",
        "subtitle": "直接看 Proposed - Original 的提升幅度。",
        "img": IMAGES["subgroup_delta"],
        "required": "subgroup metrics",
        "takeaway": "正值代表 Proposed 在該 subgroup 優於 Original。",
        "detail": """
        這張 heatmap 直接顯示 Proposed 相較 Original 的 Hit@10 改變量。
        正值代表提升，負值代表退步。

        這張圖可以用來支撐「Proposed 相較 Original 有改善」這個論點，
        並指出改善幅度最大的 subgroup。
        """
    },
    {
        "id": "hce_cases",
        "title": "High-confidence Error Examples",
        "subtitle": "作業要求至少分析 3 個 high-confidence error。",
        "img": IMAGES["hce_cases"],
        "required": "high-confidence error analysis",
        "takeaway": "這裡列出 Proposed 模型最典型的 3 個 high-confidence error。",
        "detail": """
        這張圖列出 Proposed model 的 high-confidence error 案例。
        這些案例的共同特徵是：
        模型 confidence 很高，但正確答案沒有進入 Top-10。

        這類錯誤比一般錯誤更值得注意，因為使用者可能會因為模型看起來很有把握，
        而更容易相信錯誤推薦。
        """
    },
]

image_cards_json = json.dumps(image_cards, ensure_ascii=False)


# ============================================================
# 7. 報告書 HTML
# ============================================================

def report_img(filename, caption):
    if not os.path.exists(os.path.join(OUTPUT_DIR, filename)):
        return f"<p><b>缺少圖片：</b>{escape(filename)}</p>"

    return f"""
    <figure>
      <img src="{escape(filename)}" alt="{escape(caption)}">
      <figcaption>{escape(caption)}</figcaption>
    </figure>
    """

case_report_html = ""

for case in case_cards:
    case_report_html += f"""
    <section class="case">
      <h3>{escape(case["title"])}</h3>
      <p><b>摘要：</b>{case["summary"]}</p>
      <p>{case["analysis"]}</p>
      <details>
        <summary>查看可觀測 log 摘要</summary>
        <pre>{escape(json.dumps(case["log"], ensure_ascii=False, indent=2))}</pre>
      </details>
    </section>
    """

report_html = f"""
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<title>Reliability Report</title>
<style>
body {{
  margin: 0;
  background: #f8fafc;
  color: #111827;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Microsoft JhengHei", sans-serif;
  line-height: 1.75;
}}
.container {{
  max-width: 980px;
  margin: 0 auto;
  padding: 40px 28px 80px;
}}
.header {{
  background: #111827;
  color: white;
  padding: 30px;
  border-radius: 20px;
  margin-bottom: 24px;
}}
.header h1 {{
  margin: 0 0 8px;
}}
.card, section {{
  background: white;
  padding: 24px;
  margin: 18px 0;
  border-radius: 18px;
  box-shadow: 0 3px 12px rgba(0,0,0,0.06);
}}
h2 {{
  border-bottom: 1px solid #e5e7eb;
  padding-bottom: 8px;
}}
img {{
  width: 100%;
  border: 1px solid #e5e7eb;
  border-radius: 12px;
  background: white;
}}
figcaption {{
  color: #6b7280;
  font-size: 14px;
  margin-top: 8px;
}}
pre {{
  background: #f3f4f6;
  padding: 14px;
  border-radius: 10px;
  overflow-x: auto;
}}
.print-btn {{
  position: sticky;
  top: 12px;
  float: right;
  background: #2563eb;
  color: white;
  border: none;
  border-radius: 12px;
  padding: 10px 16px;
  cursor: pointer;
  font-size: 15px;
}}
@media print {{
  .print-btn {{
    display: none;
  }}
  body {{
    background: white;
  }}
  .card, section {{
    box-shadow: none;
    border: 1px solid #e5e7eb;
    break-inside: avoid;
  }}
}}
</style>
</head>

<body>
<div class="container">

<button class="print-btn" onclick="window.print()">列印 / 另存 PDF</button>

<div class="header">
  <h1>Reliability Report：Original vs Proposed</h1>
  <p>本報告整理模型比較、可靠性分析、錯誤切片、子群體指標、高信心錯誤案例、可能錯誤來源與修正策略。</p>
</div>

<section>
  <h2>1. Executive Summary</h2>
  <p>
    本次比較 Original baseline 與 Proposed model。
    Proposed 的 Hit@10 為 <b>{pct(prop["hit10"])}</b>，
    Original 為 <b>{pct(base["hit10"])}</b>，
    差異為 <b>{pp(metric_delta_hit10)}</b>。
    Proposed 的 Median Rank 為 <b>{fmt(prop["median_rank"], 0)}</b>，
    Original 為 <b>{fmt(base["median_rank"], 0)}</b>，
    差異為 <b>{metric_delta_rank:+.0f}</b>。
  </p>
  <p>
    這表示 Proposed 在 retrieval performance 上相較 Original 有改善。
    不過 reliability 需要另外檢查。
    Proposed 的 ECE 為 <b>{fmt(prop["ece"])}</b>，
    high-confidence error rate 為 <b>{pct(prop["high_conf_error_rate"])}</b>。
    因此，結論不是單純「Proposed 比較好」，
    而是：<b>Proposed 的 retrieval performance 更好，但 confidence calibration 仍需要改善。</b>
  </p>
</section>

<section>
  <h2>2. Performance</h2>
  <p>
    Performance 比較顯示 Proposed 的 Hit@10 較 Original 提升，
    且 Median Rank 下降，代表正確答案更常被排到較前面。
    這支持 Proposed model 的設計確實對 retrieval task 有幫助。
  </p>
  {report_img(IMAGES["performance"], "Performance Comparison")}
</section>

<section>
  <h2>3. Confidence Distribution</h2>
  <p>
    Confidence distribution 顯示兩模型的信心分數分布。
    這張圖用來判斷 Proposed 是否只是更常輸出高 confidence。
    但 confidence 分布本身不能代表可靠性，
    必須搭配 Reliability Diagram 與 Confidence Band Summary 解讀。
  </p>
  {report_img(IMAGES["confidence"], "Confidence Distribution")}
</section>

<section>
  <h2>4. Reliability Diagram</h2>
  <p>
    Reliability Diagram 顯示模型 confidence 與實際 Hit@10 accuracy 是否一致。
    Proposed 的 ECE 比 Original 低，代表 calibration 略有改善；
    但若曲線仍明顯低於 perfect calibration 對角線，
    代表 Proposed 仍存在 over-confidence 問題。
  </p>
  {report_img(IMAGES["reliability"], "Reliability Diagram")}
  {report_img(IMAGES["confidence_band"], "Confidence Band Summary")}
</section>

<section>
  <h2>5. Error Slicing</h2>
  <p>
    Error slicing 顯示錯誤集中在哪些類型。
    若 Proposed 的錯誤數量在部分類型下降，代表 performance 有改善；
    但 high-confidence semantic mismatch 仍需要特別注意，
    因為這表示模型在錯誤案例中仍可能很有信心。
  </p>
  {report_img(IMAGES["error_slicing"], "Error Slicing")}
</section>

<section>
  <h2>6. Subgroup Metrics</h2>
  <p>
    Subgroup metrics 比較模型在不同條件切片下的表現：
    weather、occasion、style 與 category。
    若 Proposed 在多數 subgroup 的 Hit@10 都高於 Original，
    表示改善不是只集中在單一子群體。
    但若某些 subgroup 的 high-confidence error rate 仍高，
    則代表那些條件下仍存在 reliability risk。
  </p>
  {report_img(IMAGES["subgroup_hit10"], "Subgroup Hit@10")}
  {report_img(IMAGES["subgroup_hce"], "Subgroup High-confidence Error Rate")}
  {report_img(IMAGES["subgroup_delta"], "Subgroup ΔHit@10")}
</section>

<section>
  <h2>7. High-confidence Error Analysis</h2>
  <p>
    以下三個案例皆來自 Proposed model。
    它們的共同特徵是 confidence 高於 {HIGH_CONF_THRESHOLD}，
    但正確答案沒有進入 Top-10。
    這類錯誤比一般錯誤更重要，因為使用者更可能相信模型輸出。
  </p>
  {report_img(IMAGES["hce_cases"], "High-confidence Error Examples")}
  {case_report_html}
</section>

<section>
  <h2>8. Possible Error Sources and Fixes</h2>
  <h3>可能錯誤來源</h3>
  <ol>
    <li>
      <b>Confidence calibration 不佳：</b>
      模型 confidence 明顯高於實際 Hit@10 accuracy，造成 over-confidence。
    </li>
    <li>
      <b>語意 shortcut：</b>
      模型可能依賴部分風格詞、類別資訊或視覺相似度，
      但沒有完整理解 partial outfit 與 target item 的搭配關係。
    </li>
    <li>
      <b>候選池高度相似：</b>
      特別是 accessory-led item，候選項可能外觀相似，
      使模型容易將不正確 item 排在前面。
    </li>
  </ol>

  <h3>修正策略</h3>
  <ol>
    <li>
      <b>Confidence calibration：</b>
      使用 validation set 進行 temperature scaling 或 isotonic calibration，
      使 confidence 更接近實際正確率。
    </li>
    <li>
      <b>Reject option / uncertainty warning：</b>
      當校準後 confidence 不足或 top candidates 差距不穩定時，
      系統不要直接給單一推薦，而是提示使用者需要確認。
    </li>
    <li>
      <b>強化弱 subgroup：</b>
      針對 accessory-led 與 high-confidence mismatch 案例加入 hard negative training、
      category-aware reranking 或補充更多相似候選資料。
    </li>
  </ol>
</section>

<section>
  <h2>9. Real-world Risk Assessment</h2>
  <p>
    若這些錯誤出現在一般穿搭推薦情境，通常不屬於醫療、金融或司法等高風險決策，
    因此整體可視為 <b>低到中度風險</b>。
  </p>
  <p>
    不過 high-confidence error 不能完全視為無害。
    因為模型看起來很有信心，使用者可能更容易相信錯誤推薦。
    若錯誤發生在面試、正式會議、婚禮、極端天氣或長時間戶外活動，
    可能造成場合不合宜、穿著不舒適，甚至影響使用者對系統的信任。
  </p>
  <p>
    因此最後結論是：
    <b>Proposed model 相較 Original 有 performance improvement，
    但仍不應直接部署為完全可信的自動推薦系統；
    真實使用時需要加入 confidence calibration、uncertainty warning 與人工確認機制。</b>
  </p>
</section>

</div>
</body>
</html>
"""

with open(REPORT_HTML, "w", encoding="utf-8") as f:
    f.write(report_html)


# ============================================================
# 8. Dashboard HTML
# ============================================================

case_cards_html = ""

for case in case_cards:
    case_cards_html += f"""
    <details class="case-detail">
      <summary>{escape(case["title"])}｜{case["summary"]}</summary>
      <div class="case-body">
        <p>{case["analysis"]}</p>
        <pre>{escape(json.dumps(case["log"], ensure_ascii=False, indent=2))}</pre>
      </div>
    </details>
    """

dashboard_html = f"""
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<title>Reliability Dashboard</title>
<style>
:root {{
  --bg: #f5f6fa;
  --card: #ffffff;
  --text: #111827;
  --muted: #6b7280;
  --line: #e5e7eb;
  --blue: #2563eb;
  --blue-bg: #eff6ff;
  --green-bg: #ecfdf5;
  --yellow-bg: #fffbeb;
  --red-bg: #fef2f2;
}}
body {{
  margin: 0;
  background: var(--bg);
  color: var(--text);
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Microsoft JhengHei", sans-serif;
}}
.container {{
  max-width: 1240px;
  margin: 0 auto;
  padding: 30px 24px 70px;
}}
.header {{
  background: linear-gradient(135deg, #111827, #374151);
  color: white;
  border-radius: 24px;
  padding: 30px;
  margin-bottom: 20px;
}}
.header h1 {{
  margin: 0 0 8px;
  font-size: 34px;
}}
.header p {{
  margin: 0;
  color: #e5e7eb;
}}
.top-actions {{
  display: flex;
  gap: 12px;
  flex-wrap: wrap;
  margin-top: 20px;
}}
.btn {{
  border: none;
  border-radius: 12px;
  padding: 12px 18px;
  font-size: 15px;
  cursor: pointer;
  text-decoration: none;
  display: inline-block;
}}
.btn-primary {{
  background: #2563eb;
  color: white;
}}
.btn-secondary {{
  background: white;
  color: #111827;
}}
.guide {{
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 14px;
  margin: 18px 0;
}}
.guide-card {{
  background: white;
  border-radius: 18px;
  padding: 18px;
  box-shadow: 0 3px 12px rgba(0,0,0,0.05);
}}
.guide-card b {{
  display: block;
  margin-bottom: 6px;
}}
.summary-grid {{
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 14px;
  margin: 18px 0;
}}
.metric {{
  background: white;
  border-radius: 18px;
  padding: 18px;
  box-shadow: 0 3px 12px rgba(0,0,0,0.05);
}}
.metric-label {{
  color: var(--muted);
  font-size: 14px;
}}
.metric-value {{
  font-size: 28px;
  font-weight: 700;
  margin-top: 8px;
}}
.metric-note {{
  color: var(--muted);
  font-size: 13px;
  margin-top: 6px;
}}
.section-title {{
  margin: 28px 0 12px;
}}
.gallery {{
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 18px;
}}
.figure-card {{
  background: white;
  border-radius: 20px;
  overflow: hidden;
  box-shadow: 0 4px 16px rgba(0,0,0,0.06);
  cursor: pointer;
  transition: transform 0.15s ease, box-shadow 0.15s ease;
}}
.figure-card:hover {{
  transform: translateY(-2px);
  box-shadow: 0 8px 24px rgba(0,0,0,0.10);
}}
.figure-card img {{
  width: 100%;
  height: 210px;
  object-fit: contain;
  background: white;
  border-bottom: 1px solid var(--line);
}}
.figure-card-body {{
  padding: 16px;
}}
.figure-card h3 {{
  margin: 0 0 6px;
  font-size: 18px;
}}
.figure-card p {{
  margin: 0;
  color: var(--muted);
  font-size: 14px;
  line-height: 1.5;
}}
.badge {{
  display: inline-block;
  font-size: 12px;
  background: var(--blue-bg);
  color: #1d4ed8;
  border-radius: 999px;
  padding: 4px 9px;
  margin-top: 10px;
}}
.card {{
  background: white;
  border-radius: 20px;
  padding: 22px;
  margin: 18px 0;
  box-shadow: 0 4px 16px rgba(0,0,0,0.06);
}}
.checklist {{
  display: grid;
  grid-template-columns: repeat(2, 1fr);
  gap: 12px;
}}
.check-item {{
  background: var(--green-bg);
  border-left: 5px solid #10b981;
  padding: 14px;
  border-radius: 12px;
}}
.case-detail {{
  background: white;
  border: 1px solid var(--line);
  border-radius: 14px;
  margin: 12px 0;
  padding: 14px 16px;
}}
.case-detail summary {{
  cursor: pointer;
  font-weight: 600;
}}
.case-body {{
  margin-top: 12px;
  line-height: 1.75;
}}
pre {{
  background: #f3f4f6;
  border-radius: 12px;
  padding: 14px;
  overflow-x: auto;
}}
.risk-box {{
  background: var(--red-bg);
  border-left: 5px solid #dc2626;
  border-radius: 14px;
  padding: 16px;
  line-height: 1.8;
}}
.modal {{
  display: none;
  position: fixed;
  z-index: 20;
  inset: 0;
  background: rgba(17, 24, 39, 0.72);
  padding: 30px;
}}
.modal-content {{
  background: white;
  border-radius: 24px;
  max-width: 1100px;
  max-height: 90vh;
  overflow: auto;
  margin: 0 auto;
  padding: 24px;
}}
.modal-head {{
  display: flex;
  justify-content: space-between;
  align-items: start;
  gap: 16px;
}}
.close {{
  border: none;
  background: #f3f4f6;
  border-radius: 10px;
  padding: 8px 12px;
  cursor: pointer;
}}
.modal img {{
  width: 100%;
  max-height: 540px;
  object-fit: contain;
  background: white;
  border: 1px solid var(--line);
  border-radius: 14px;
  margin: 16px 0;
}}
.modal-text {{
  line-height: 1.8;
  background: var(--blue-bg);
  border-left: 5px solid #2563eb;
  padding: 16px;
  border-radius: 14px;
}}
@media (max-width: 1000px) {{
  .guide, .summary-grid, .gallery, .checklist {{
    grid-template-columns: 1fr 1fr;
  }}
}}
@media (max-width: 700px) {{
  .guide, .summary-grid, .gallery, .checklist {{
    grid-template-columns: 1fr;
  }}
}}
</style>
</head>

<body>
<div class="container">

  <div class="header">
    <h1>Reliability Dashboard｜Original vs Proposed</h1>
    <p>比較兩個模型的 performance 與 reliability，並提供 high-confidence error 分析與報告書輸出。</p>
    <div class="top-actions">
      <a class="btn btn-primary" href="reliability_report.html" target="_blank">輸出報告書</a>
      <button class="btn btn-secondary" onclick="scrollToGallery()">查看圖表</button>
    </div>
  </div>

  <div class="guide">
    <div class="guide-card">
      <b>1. 先看總覽</b>
      從 Hit@10、Median Rank、ECE 判斷 Proposed 是否比 Original 更好。
    </div>
    <div class="guide-card">
      <b>2. 點選圖片</b>
      圖片集中在下方圖卡區，點一下即可查看完整解釋與解讀方式。
    </div>
    <div class="guide-card">
      <b>3. 看錯誤案例</b>
      往下展開 3 個 high-confidence error，檢查模型為什麼「很有信心但錯」。
    </div>
    <div class="guide-card">
      <b>4. 輸出報告</b>
      點選「輸出報告書」會開啟完整報告，可再列印或另存 PDF。
    </div>
  </div>

  <div class="summary-grid">
    <div class="metric">
      <div class="metric-label">Hit@10 improvement</div>
      <div class="metric-value">{pp(metric_delta_hit10)}</div>
      <div class="metric-note">Proposed {pct(prop["hit10"])} vs Original {pct(base["hit10"])}</div>
    </div>
    <div class="metric">
      <div class="metric-label">Median Rank change</div>
      <div class="metric-value">{metric_delta_rank:+.0f}</div>
      <div class="metric-note">負值代表 Proposed 排名更前面</div>
    </div>
    <div class="metric">
      <div class="metric-label">ECE change</div>
      <div class="metric-value">{metric_delta_ece:+.3f}</div>
      <div class="metric-note">負值代表 calibration 變好</div>
    </div>
    <div class="metric">
      <div class="metric-label">High-conf Error change</div>
      <div class="metric-value">{pp(metric_delta_hce)}</div>
      <div class="metric-note">負值代表高信心錯誤下降</div>
    </div>
  </div>

  <div class="card">
    <h2>作業要求檢查</h2>
    <div class="checklist">
      <div class="check-item">已報告 performance：見 Performance Comparison。</div>
      <div class="check-item">已報告 confidence distribution：見 Confidence Distribution。</div>
      <div class="check-item">已報告 error slicing：見 Error Slicing。</div>
      <div class="check-item">已報告 subgroup metrics：見 Subgroup Metrics。</div>
      <div class="check-item">已分析 3 個 high-confidence error：見下方案例分析。</div>
      <div class="check-item">已提出錯誤來源、修正策略與真實風險判斷。</div>
    </div>
  </div>

  <h2 class="section-title" id="gallery-title">圖表區：點選圖片查看說明</h2>
  <div class="gallery" id="gallery"></div>

  <div class="card">
    <h2>3 個 High-confidence Error 分析</h2>
    <p>
      以下案例皆來自 Proposed model。
      它們的共同條件是 confidence ≥ {HIGH_CONF_THRESHOLD}，
      但正確答案沒有進入 Top-10。
    </p>
    {case_cards_html}
  </div>

  <div class="card">
    <h2>可能錯誤來源與修正策略</h2>
    <div class="checklist">
      <div class="check-item">
        <b>錯誤來源 1：confidence calibration 不佳</b><br>
        模型 confidence 高於實際 Hit@10 accuracy，造成 over-confidence。
      </div>
      <div class="check-item">
        <b>修正策略 1：confidence calibration</b><br>
        使用 validation set 做 temperature scaling 或 isotonic calibration。
      </div>
      <div class="check-item">
        <b>錯誤來源 2：語意 shortcut / 候選池相似</b><br>
        模型可能依賴局部風格詞、類別或視覺相似度，沒有完整理解搭配關係。
      </div>
      <div class="check-item">
        <b>修正策略 2：hard negative + reranking</b><br>
        對 accessory-led 與 high-confidence mismatch 案例加強 hard negative training 或 category-aware reranking。
      </div>
    </div>
  </div>

  <div class="card">
    <h2>真實使用情境風險判斷</h2>
    <div class="risk-box">
      若錯誤出現在一般穿搭推薦情境，通常不屬於醫療、金融或司法等高風險決策，
      因此整體是 <b>低到中度風險</b>。
      但 high-confidence error 仍需要注意，因為使用者可能因模型看起來很有把握而過度相信錯誤推薦。
      若發生在正式場合、面試、婚禮、極端天氣或長時間戶外活動，
      可能造成場合不合宜或穿著不舒適。
      因此 Proposed 雖然相較 Original 有改善，真實部署時仍應加入
      <b>confidence calibration、uncertainty warning 與人工確認機制</b>。
    </div>
  </div>

</div>

<div class="modal" id="modal">
  <div class="modal-content">
    <div class="modal-head">
      <div>
        <h2 id="modal-title"></h2>
        <p id="modal-subtitle"></p>
      </div>
      <button class="close" onclick="closeModal()">關閉</button>
    </div>
    <img id="modal-img" src="">
    <div class="modal-text">
      <h3>如何解讀</h3>
      <p id="modal-takeaway"></p>
      <div id="modal-detail"></div>
    </div>
  </div>
</div>

<script>
const cards = {image_cards_json};

function renderGallery() {{
  const gallery = document.getElementById("gallery");
  gallery.innerHTML = "";

  cards.forEach((card, idx) => {{
    const div = document.createElement("div");
    div.className = "figure-card";
    div.onclick = () => openModal(idx);

    div.innerHTML = `
      <img src="${{card.img}}" alt="${{card.title}}">
      <div class="figure-card-body">
        <h3>${{card.title}}</h3>
        <p>${{card.subtitle}}</p>
        <span class="badge">${{card.required}}</span>
      </div>
    `;

    gallery.appendChild(div);
  }});
}}

function openModal(idx) {{
  const card = cards[idx];

  document.getElementById("modal-title").innerText = card.title;
  document.getElementById("modal-subtitle").innerText = card.subtitle;
  document.getElementById("modal-img").src = card.img;
  document.getElementById("modal-takeaway").innerText = card.takeaway;
  document.getElementById("modal-detail").innerHTML = card.detail;

  document.getElementById("modal").style.display = "block";
}}

function closeModal() {{
  document.getElementById("modal").style.display = "none";
}}

function scrollToGallery() {{
  document.getElementById("gallery-title").scrollIntoView({{ behavior: "smooth" }});
}}

window.onclick = function(event) {{
  const modal = document.getElementById("modal");
  if (event.target === modal) {{
    closeModal();
  }}
}}

renderGallery();
</script>

</body>
</html>
"""

with open(DASHBOARD_HTML, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("已輸出 dashboard：", DASHBOARD_HTML)
print("已輸出報告書：", REPORT_HTML)
print("Dashboard 內的「輸出報告書」按鍵會開啟 reliability_report.html")

已輸出 dashboard： ./reliability_outputs/compare/reliability_dashboard.html
已輸出報告書： ./reliability_outputs/compare/reliability_report.html
Dashboard 內的「輸出報告書」按鍵會開啟 reliability_report.html


In [28]:
# ============================================================
# 02_reliability_dashboard.py
#
# 功能：
# 1. 重新整理 dashboard 版面：圖片集中成圖卡
# 2. 點選圖片才顯示詳細說明
# 3. 首頁顯示「如何查看資料」引導
# 4. 自動整理：
#    - performance
#    - confidence distribution
#    - error slicing
#    - subgroup metrics
#    - 3 個 high-confidence error 分析
#    - 可能錯誤來源與修正策略
#    - real-world risk assessment
# 5. 一個按鍵輸出報告書 reliability_report.html
#
# 輸入：
# - reliability_required_fields.csv
# - plot_*.png
#
# 輸出：
# - reliability_dashboard.html
# - reliability_report.html
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from html import escape


# ============================================================
# 1. 路徑設定
# ============================================================

OUTPUT_DIR = "./reliability_outputs/compare"

REQUIRED_FIELDS_CSV = os.path.join(OUTPUT_DIR, "reliability_required_fields.csv")
DASHBOARD_HTML = os.path.join(OUTPUT_DIR, "reliability_dashboard.html")
REPORT_HTML = os.path.join(OUTPUT_DIR, "reliability_report.html")

BASELINE_TAG = "original"
PROPOSED_TAG = "proposed"
HIGH_CONF_THRESHOLD = 0.85

if not os.path.exists(REQUIRED_FIELDS_CSV):
    raise FileNotFoundError(f"找不到必要 CSV：{REQUIRED_FIELDS_CSV}")


# ============================================================
# 2. 讀取資料
# ============================================================

df = pd.read_csv(REQUIRED_FIELDS_CSV)

numeric_cols = [
    "confidence",
    "rank",
    "hit@1",
    "hit@3",
    "hit@5",
    "hit@10",
    "hit@30",
    "hit@50",
    "is_error",
    "is_high_conf_error",
    "top1_score",
    "top2_score",
    "target_score",
    "top1_top2_gap",
    "temperature_c",
    "style_count",
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["hit@10"] = df["hit@10"].fillna(0).astype(int)
df["is_error"] = (df["hit@10"] == 0).astype(int)
df["is_high_conf_error"] = (
    (df["confidence"] >= HIGH_CONF_THRESHOLD) & (df["is_error"] == 1)
).astype(int)

if "metadata_key" not in df.columns:
    df["metadata_key"] = (
        df["seed"].astype(str)
        + "||"
        + df["set_id"].astype(str)
        + "||"
        + df["target_item_id"].astype(str)
    )


# ============================================================
# 3. 指標函式
# ============================================================

def pct(x):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:.2f}%"

def pp(x):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:+.2f} pp"

def fmt(x, digits=3):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

def compute_ece(data, n_bins=10):
    temp = data[["confidence", "hit@10"]].dropna().copy()

    if len(temp) == 0:
        return np.nan

    temp["confidence"] = temp["confidence"].clip(0, 1)
    bins = np.linspace(0, 1, n_bins + 1)
    temp["bin"] = pd.cut(temp["confidence"], bins=bins, include_lowest=True)

    ece = 0.0

    for _, g in temp.groupby("bin", observed=False):
        if len(g) == 0:
            continue

        avg_conf = g["confidence"].mean()
        acc = g["hit@10"].mean()
        weight = len(g) / len(temp)
        ece += weight * abs(avg_conf - acc)

    return float(ece)

def brier_score(data):
    temp = data[["confidence", "hit@10"]].dropna().copy()

    if len(temp) == 0:
        return np.nan

    return float(np.mean((temp["confidence"].clip(0, 1) - temp["hit@10"]) ** 2))

def summarize_one_run(g):
    return {
        "n": len(g),
        "hit1": g["hit@1"].mean(),
        "hit10": g["hit@10"].mean(),
        "median_rank": g["rank"].median(),
        "mean_rank": g["rank"].mean(),
        "avg_confidence": g["confidence"].mean(),
        "error_rate": g["is_error"].mean(),
        "high_conf_error_rate": g["is_high_conf_error"].mean(),
        "ece": compute_ece(g),
        "brier": brier_score(g),
    }

summary = {}

for run_tag, g in df.groupby("run_tag"):
    summary[run_tag] = summarize_one_run(g)

base = summary.get(BASELINE_TAG)
prop = summary.get(PROPOSED_TAG)

if base is None or prop is None:
    raise ValueError("CSV 中需要同時包含 run_tag = original 與 run_tag = proposed。")


# ============================================================
# 4. 圖片檔案自動偵測
# ============================================================

def first_existing(candidates):
    for fname in candidates:
        if os.path.exists(os.path.join(OUTPUT_DIR, fname)):
            return fname
    return candidates[0]

IMAGES = {
    "performance": first_existing([
        "plot_01_performance_comparison.png",
        "plot_01_performance_summary.png",
    ]),
    "confidence": first_existing([
        "plot_02_confidence_distribution_comparison.png",
        "plot_02_confidence_distribution.png",
    ]),
    "reliability": first_existing([
        "plot_03_reliability_diagram_comparison.png",
        "plot_03_reliability_diagram.png",
    ]),
    "confidence_band": first_existing([
        "plot_04_confidence_band_comparison.png",
        "plot_07_confidence_band_table.png",
    ]),
    "error_slicing": first_existing([
        "plot_05_error_slicing_comparison.png",
        "plot_04_error_slicing.png",
    ]),
    "subgroup_hit10": first_existing([
        "plot_06_subgroup_hit10_comparison.png",
        "plot_05_subgroup_hit10.png",
    ]),
    "subgroup_hce": first_existing([
        "plot_07_subgroup_high_conf_error_comparison.png",
        "plot_06_subgroup_high_conf_error.png",
    ]),
    "subgroup_delta": first_existing([
        "plot_08_subgroup_delta_hit10_heatmap.png",
        "plot_10_subset_delta_heatmap.png",
    ]),
    "hce_cases": first_existing([
        "plot_09_high_confidence_errors_comparison.png",
        "plot_08_high_confidence_errors.png",
    ]),
}


# ============================================================
# 5. High-confidence error cases
# ============================================================

def safe_value(row, col, default=""):
    if col not in row:
        return default
    v = row[col]
    if pd.isna(v):
        return default
    return v

prop_df = df[df["run_tag"] == PROPOSED_TAG].copy()
base_df = df[df["run_tag"] == BASELINE_TAG].copy()

base_compare_cols = [
    "metadata_key",
    "model_output_top1_id",
    "rank",
    "confidence",
    "hit@10",
]

base_compare = base_df[base_compare_cols].rename(columns={
    "model_output_top1_id": "original_top1_id",
    "rank": "original_rank",
    "confidence": "original_confidence",
    "hit@10": "original_hit10",
})

hce = (
    prop_df[prop_df["is_high_conf_error"] == 1]
    .sort_values(["confidence", "rank"], ascending=[False, False])
    .head(3)
    .merge(base_compare, on="metadata_key", how="left")
)

def build_case_analysis(row, idx):
    rank = safe_value(row, "rank", np.nan)
    original_rank = safe_value(row, "original_rank", np.nan)
    confidence = safe_value(row, "confidence", np.nan)
    original_conf = safe_value(row, "original_confidence", np.nan)

    if pd.notna(original_rank):
        rank_change = rank - original_rank

        if rank_change < 0:
            compare_sentence = (
                f"相較 Original，Proposed 的正確答案排名前進了 <b>{abs(rank_change):.0f} 名</b>，"
                f"但 rank 仍為 <b>{rank:.0f}</b>，沒有進入 Top-10。"
            )
        elif rank_change > 0:
            compare_sentence = (
                f"相較 Original，Proposed 的正確答案排名退後了 <b>{rank_change:.0f} 名</b>，"
                f"表示這個案例中 Proposed 雖然信心高，但排序反而更差。"
            )
        else:
            compare_sentence = (
                "Proposed 與 Original 的正確答案排名相同，但仍沒有進入 Top-10。"
            )
    else:
        compare_sentence = "此案例沒有找到可對應的 Original 結果，因此主要分析 Proposed 本身的錯誤。"

    category = safe_value(row, "fine_category", "unknown")
    major = safe_value(row, "major_category", "unknown")
    weather = safe_value(row, "weather_group", "unknown")
    occasion = safe_value(row, "occasion_group", "unknown")
    style = safe_value(row, "style_group", "unknown")
    error_type = safe_value(row, "error_type", "unknown")
    source = safe_value(row, "possible_source", "")

    return {
        "title": f"High-confidence Error Case {idx}",
        "summary": (
            f"Proposed confidence = {fmt(confidence, 3)}，"
            f"rank = {fmt(rank, 0)}，"
            f"target = {escape(str(safe_value(row, 'target_item_id')))}，"
            f"top1 = {escape(str(safe_value(row, 'model_output_top1_id')))}。"
        ),
        "analysis": f"""
        這是一個 high-confidence error：Proposed 模型給出非常高的 confidence，
        但正確答案沒有進入 Top-10。<br><br>{compare_sentence}<br><br>
        這表示 Proposed 即使在 performance 上相較 Original 有改善，仍可能在個別案例中非常有信心地排錯。<br><br>

        此案例的 target item 類別為 <span class="badge badge-light">{escape(str(category))}</span>，
        major category 為 <span class="badge badge-light">{escape(str(major))}</span>；
        subgroup 為 weather = <b>{escape(str(weather))}</b>、
        occasion = <b>{escape(str(occasion))}</b>、
        style = <b>{escape(str(style))}</b>。<br>
        錯誤類型被歸為 <b>{escape(str(error_type))}</b>。<br><br>

        <b>可能來源：</b>{escape(str(source))}
        """,
        "log": {
            "set_id": safe_value(row, "set_id"),
            "correct_answer": safe_value(row, "correct_answer"),
            "proposed_top1": safe_value(row, "model_output_top1_id"),
            "original_top1": safe_value(row, "original_top1_id"),
            "proposed_rank": safe_value(row, "rank"),
            "original_rank": safe_value(row, "original_rank"),
            "proposed_confidence": safe_value(row, "confidence"),
            "original_confidence": safe_value(row, "original_confidence"),
            "error_type": safe_value(row, "error_type"),
            "risk_level": safe_value(row, "risk_level"),
        }
    }

case_cards = [
    build_case_analysis(row, i + 1)
    for i, (_, row) in enumerate(hce.iterrows())
]


# ============================================================
# 6. 圖片卡說明
# ============================================================

metric_delta_hit10 = prop["hit10"] - base["hit10"]
metric_delta_rank = prop["median_rank"] - base["median_rank"]
metric_delta_ece = prop["ece"] - base["ece"]
metric_delta_hce = prop["high_conf_error_rate"] - base["high_conf_error_rate"]

image_cards = [
    {
        "id": "performance",
        "title": "Performance Comparison",
        "subtitle": "先看 Proposed 是否比 Original 更準。",
        "img": IMAGES["performance"],
        "required": "performance",
        "takeaway": (
            f"Proposed Hit@10 = {pct(prop['hit10'])}，Original = {pct(base['hit10'])}，"
            f"差異 = {pp(metric_delta_hit10)}。Median Rank 差異 = {metric_delta_rank:+.0f}。"
        ),
        "detail": """
        這張圖是整份 dashboard 的起點。它比較 Original baseline 與 Proposed model 的整體表現。<br>
        閱讀時先看 <b>Hit@10 是否上升</b>，再看 <b>Median Rank 是否下降</b>。<br>
        Hit@10 上升代表正確答案更常進入前 10 名；Median Rank 下降代表正確答案整體被排得更前面。<br><br>
        這裡也要同時看 <b>ECE</b> 與 <b>High-confidence Error Rate</b>。
        如果 Proposed 的 Hit@10 變好，但 ECE 仍高，代表它雖然更準，但 confidence 仍不一定能解讀成「真的正確的機率」。
        """
    },
    {
        "id": "confidence",
        "title": "Confidence Distribution",
        "subtitle": "比較兩模型信心分數分布。",
        "img": IMAGES["confidence"],
        "required": "confidence distribution",
        "takeaway": (
            f"Original 平均 confidence = {fmt(base['avg_confidence'])}，"
            f"Proposed = {fmt(prop['avg_confidence'])}。"
        ),
        "detail": """
        這張圖用來看兩個模型是否常輸出高 confidence。
        不過 confidence distribution 本身不能代表模型可靠。
        一個模型可以經常輸出高 confidence，但實際上仍然常錯。<br><br>
        因此，這張圖要和 <b>Reliability Diagram</b> 以及 <b>Confidence Band Summary</b> 一起看。
        如果 Proposed 的 confidence 分布與 Original 類似，但 Hit@10 提升，
        代表 performance 改善不是單純來自更高的 confidence。
        """
    },
    {
        "id": "reliability",
        "title": "Reliability Diagram",
        "subtitle": "檢查 confidence 是否校準。",
        "img": IMAGES["reliability"],
        "required": "confidence calibration",
        "takeaway": (
            f"Original ECE = {fmt(base['ece'])}，Proposed ECE = {fmt(prop['ece'])}，"
            f"差異 = {metric_delta_ece:+.3f}。"
        ),
        "detail": """
        Reliability Diagram 用來比較 confidence 與實際 Hit@10 accuracy 是否一致。
        理想情況下，模型曲線應接近對角線。<br>
        若曲線明顯低於對角線，代表模型 <b>over-confident</b>：它給出的 confidence 高於實際正確率。<br><br>
        如果 Proposed 的 ECE 比 Original 低，代表 calibration 有些改善；
        但只要曲線仍然遠低於對角線，就表示 Proposed 仍需要 confidence calibration。
        """
    },
    {
        "id": "confidence_band",
        "title": "Confidence Band Summary",
        "subtitle": "看高 confidence 區間是否真的比較準。",
        "img": IMAGES["confidence_band"],
        "required": "confidence distribution",
        "takeaway": "重點看 0.90–1.00 區間：高信心是否仍然高錯誤率。",
        "detail": """
        這張表把樣本依 confidence 區間分組，直接檢查高信心區間是否對應較高 Hit@10。
        如果 0.90–1.00 區間的 Hit@10 仍偏低，
        或 high-confidence error rate 很高，就代表模型在高信心下仍可能錯。<br><br>
        這張圖是挑選 high-confidence error 進行個案分析的主要依據。
        """
    },
    {
        "id": "error_slicing",
        "title": "Error Slicing",
        "subtitle": "比較錯誤集中在哪些類型。",
        "img": IMAGES["error_slicing"],
        "required": "error slicing",
        "takeaway": "觀察 Proposed 的錯誤是否比 Original 少，以及哪些錯誤仍最多。",
        "detail": """
        Error slicing 比較 Original 與 Proposed 的錯誤類型分布。<br>
        如果 Proposed 的總錯誤數下降，代表 performance 有改善；
        但如果 <b>high-confidence semantic mismatch</b> 仍然很多，代表 reliability failure 仍然存在。<br><br>
        特別是 <b>accessory-led retrieval failure</b> 若仍是主要錯誤來源，
        表示模型在配件類 target item 上仍比較弱。
        """
    },
    {
        "id": "subgroup_hit10",
        "title": "Subgroup Metrics: Hit@10",
        "subtitle": "看不同子群體是否都有改善。",
        "img": IMAGES["subgroup_hit10"],
        "required": "subgroup metrics",
        "takeaway": "比較 cold/warm、formal/casual、style、category 各群體的 Hit@10。",
        "detail": """
        這張圖比較兩個模型在不同 subgroup 上的 Hit@10。<br>
        如果 Proposed 在多數 subgroup 都高於 Original，
        代表改善不是只來自單一子群體，而是較全面的提升。<br><br>
        如果某個 subgroup 仍然偏低，例如 <b>accessory-led</b>，
        代表該類別仍是模型弱點，需要後續針對性修正。
        """
    },
    {
        "id": "subgroup_hce",
        "title": "Subgroup Metrics: High-confidence Error Rate",
        "subtitle": "看哪個子群體最容易「很有信心但錯」。",
        "img": IMAGES["subgroup_hce"],
        "required": "subgroup metrics",
        "takeaway": (
            f"Proposed high-confidence error rate = {pct(prop['high_conf_error_rate'])}，"
            f"Original = {pct(base['high_conf_error_rate'])}。"
        ),
        "detail": """
        這張圖比 Hit@10 更接近 reliability 的角度，
        因為它直接觀察「模型很有信心卻仍然錯」的比例。<br><br>
        如果 Proposed 在某些 subgroup 的 high-confidence error rate 沒有下降，
        代表該群體雖然可能 performance 有改善，但使用者仍可能過度相信錯誤推薦。
        """
    },
    {
        "id": "subgroup_delta",
        "title": "Subgroup ΔHit@10 Heatmap",
        "subtitle": "直接看 Proposed - Original 的提升幅度。",
        "img": IMAGES["subgroup_delta"],
        "required": "subgroup metrics",
        "takeaway": "正值代表 Proposed 在該 subgroup 優於 Original。",
        "detail": """
        這張 heatmap 直接顯示 Proposed 相較 Original 的 Hit@10 改變量。
        正值代表提升，負值代表退步。<br><br>
        這張圖可以用來支撐「Proposed 相較 Original 有改善」這個論點，
        並指出改善幅度最大的 subgroup。
        """
    },
    {
        "id": "hce_cases",
        "title": "High-confidence Error Examples",
        "subtitle": "作業要求至少分析 3 個 high-confidence error。",
        "img": IMAGES["hce_cases"],
        "required": "high-confidence error analysis",
        "takeaway": "這裡列出 Proposed 模型最典型的 3 個 high-confidence error。",
        "detail": """
        這張圖列出 Proposed model 的 high-confidence error 案例。<br>
        這些案例的共同特徵是：<b>模型 confidence 很高，但正確答案沒有進入 Top-10</b>。<br><br>
        這類錯誤比一般錯誤更值得注意，因為使用者可能會因為模型看起來很有把握，
        而更容易相信錯誤推薦。
        """
    },
]

image_cards_json = json.dumps(image_cards, ensure_ascii=False)


# ============================================================
# 7. 報告書 HTML (包含列印彩色優化)
# ============================================================

def report_img(filename, caption):
    if not os.path.exists(os.path.join(OUTPUT_DIR, filename)):
        return f'<div class="alert alert-warning"><b>缺少圖片：</b>{escape(filename)}</div>'

    return f"""
    <figure>
      <img src="{escape(filename)}" alt="{escape(caption)}">
      <figcaption>{escape(caption)}</figcaption>
    </figure>
    """

case_report_html = ""

for case in case_cards:
    case_report_html += f"""
    <div class="case-card">
      <h3 class="case-title">{escape(case["title"])}</h3>
      <div class="alert alert-info"><b>摘要：</b>{case["summary"]}</div>
      <p class="case-desc">{case["analysis"]}</p>
      <details class="case-log">
        <summary>查看可觀測 log 摘要</summary>
        <pre>{escape(json.dumps(case["log"], ensure_ascii=False, indent=2))}</pre>
      </details>
    </div>
    """

report_html = f"""
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<title>Reliability Report: Original vs Proposed</title>
<style>
:root {{
  --primary: #3b82f6;
  --primary-dark: #2563eb;
  --success: #10b981;
  --warning: #f59e0b;
  --danger: #ef4444;
  --bg-body: #f8fafc;
  --bg-card: #ffffff;
  --text-main: #334155;
  --text-muted: #64748b;
  --border: #e2e8f0;
}}

/* 確保列印時保留色彩背景 (Chrome, Safari, Edge, Firefox) */
@media print {{
  * {{
    -webkit-print-color-adjust: exact !important;
    print-color-adjust: exact !important;
  }}
  body {{ background: #ffffff !important; }}
  .container {{ max-width: 100% !important; padding: 0 20px !important; margin: 0 !important; }}
  .print-btn {{ display: none !important; }}
  section {{ box-shadow: none !important; border: 1px solid #e2e8f0 !important; break-inside: avoid; margin-bottom: 24px !important; }}
  .case-card {{ break-inside: avoid; }}
}}

body {{
  margin: 0; background: var(--bg-body); color: var(--text-main);
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
  line-height: 1.7; -webkit-font-smoothing: antialiased;
}}
.container {{ max-width: 960px; margin: 0 auto; padding: 40px 24px 80px; }}

.header {{
  background: linear-gradient(135deg, #1e293b, #334155); color: white;
  padding: 40px; border-radius: 20px; margin-bottom: 32px; box-shadow: 0 10px 25px -5px rgba(0,0,0,0.1);
}}
.header h1 {{ margin: 0 0 12px; font-size: 32px; font-weight: 700; letter-spacing: -0.5px; }}
.header p {{ margin: 0; color: #cbd5e1; font-size: 16px; max-width: 800px; }}

section {{
  background: var(--bg-card); padding: 32px; margin: 24px 0;
  border-radius: 16px; border: 1px solid var(--border); box-shadow: 0 4px 6px -1px rgba(0,0,0,0.02);
}}
h2 {{
  margin-top: 0; color: #0f172a; font-size: 22px; padding-bottom: 12px;
  border-bottom: 2px solid #f1f5f9; display: flex; align-items: center; gap: 10px;
}}
h2::before {{
  content: ""; display: block; width: 5px; height: 22px; background: var(--primary); border-radius: 4px;
}}
h3 {{ color: #1e293b; font-size: 18px; margin-top: 24px; }}

.alert {{ padding: 16px 20px; border-radius: 12px; margin: 16px 0; border-left: 4px solid; }}
.alert-info {{ background: #eff6ff; border-left-color: var(--primary); color: #1e3a8a; }}
.alert-warning {{ background: #fffbeb; border-left-color: var(--warning); color: #92400e; }}
.alert-danger {{ background: #fef2f2; border-left-color: var(--danger); color: #991b1b; }}
.alert-success {{ background: #f0fdf4; border-left-color: var(--success); color: #166534; }}

figure {{ margin: 24px 0; text-align: center; }}
img {{ width: 100%; border: 1px solid var(--border); border-radius: 12px; background: white; }}
figcaption {{ color: var(--text-muted); font-size: 14px; margin-top: 10px; font-weight: 500; }}

ol, ul {{ padding-left: 24px; }}
li {{ margin-bottom: 12px; }}
li::marker {{ font-weight: bold; color: var(--primary); }}

.case-card {{ background: #f8fafc; border: 1px solid var(--border); border-radius: 16px; padding: 24px; margin-top: 20px; }}
.case-title {{ margin: 0 0 16px 0; color: #0f172a; border-bottom: none; }}
.case-desc {{ margin-bottom: 16px; }}
pre {{ background: #1e293b; color: #f8fafc; padding: 16px; border-radius: 10px; overflow-x: auto; font-size: 13px; }}
.badge {{ display: inline-block; padding: 2px 8px; border-radius: 6px; font-size: 13px; font-weight: 600; background: #e2e8f0; color: #475569; }}

.print-btn {{
  position: sticky; top: 20px; float: right; background: var(--primary); color: white;
  border: none; border-radius: 10px; padding: 10px 20px; font-weight: 600;
  cursor: pointer; font-size: 14px; box-shadow: 0 4px 6px -1px rgba(59,130,246,0.3); transition: background 0.2s;
}}
.print-btn:hover {{ background: var(--primary-dark); }}

b {{ color: #0f172a; font-weight: 600; }}
</style>
</head>

<body>
<div class="container">

<button class="print-btn" onclick="window.print()">列印 / 另存 PDF</button>

<div class="header">
  <h1>Reliability Report：Original vs Proposed</h1>
  <p>本報告整理模型比較、可靠性分析、錯誤切片、子群體指標、高信心錯誤案例、可能錯誤來源與修正策略。</p>
</div>

<section>
  <h2>1. Executive Summary</h2>
  <p>
    本次比較 Original baseline 與 Proposed model。
    Proposed 的 Hit@10 為 <b>{pct(prop["hit10"])}</b>，
    Original 為 <b>{pct(base["hit10"])}</b>，
    差異為 <b>{pp(metric_delta_hit10)}</b>。
    Proposed 的 Median Rank 為 <b>{fmt(prop["median_rank"], 0)}</b>，
    Original 為 <b>{fmt(base["median_rank"], 0)}</b>，
    差異為 <b>{metric_delta_rank:+.0f}</b>。
  </p>
  <p>
    這表示 Proposed 在 retrieval performance 上相較 Original 有改善。
    不過 reliability 需要另外檢查。
    Proposed 的 ECE 為 <b>{fmt(prop["ece"])}</b>，
    high-confidence error rate 為 <b>{pct(prop["high_conf_error_rate"])}</b>。
  </p>
  <div class="alert alert-info">
    <b>核心結論：</b>Proposed 的 retrieval performance 更好，但 confidence calibration 仍需要顯著改善。
  </div>
</section>

<section>
  <h2>2. Performance</h2>
  <p>
    Performance 比較顯示 Proposed 的 Hit@10 較 Original 提升，
    且 Median Rank 下降，代表正確答案更常被排到較前面。
    這支持 Proposed model 的設計確實對 retrieval task 有幫助。
  </p>
  {report_img(IMAGES["performance"], "Performance Comparison")}
</section>

<section>
  <h2>3. Confidence Distribution</h2>
  <p>
    Confidence distribution 顯示兩模型的信心分數分布。
    這張圖用來判斷 Proposed 是否只是更常輸出高 confidence。
    但 confidence 分布本身不能代表可靠性，
    必須搭配 Reliability Diagram 與 Confidence Band Summary 解讀。
  </p>
  {report_img(IMAGES["confidence"], "Confidence Distribution")}
</section>

<section>
  <h2>4. Reliability Diagram</h2>
  <p>
    Reliability Diagram 顯示模型 confidence 與實際 Hit@10 accuracy 是否一致。
    Proposed 的 ECE 比 Original 低，代表 calibration 略有改善；
    但若曲線仍明顯低於 perfect calibration 對角線，
    代表 Proposed 仍存在 over-confidence 問題。
  </p>
  {report_img(IMAGES["reliability"], "Reliability Diagram")}
  {report_img(IMAGES["confidence_band"], "Confidence Band Summary")}
</section>

<section>
  <h2>5. Error Slicing</h2>
  <p>
    Error slicing 顯示錯誤集中在哪些類型。
    若 Proposed 的錯誤數量在部分類型下降，代表 performance 有改善；
    但 high-confidence semantic mismatch 仍需要特別注意，
    因為這表示模型在錯誤案例中仍可能很有信心。
  </p>
  {report_img(IMAGES["error_slicing"], "Error Slicing")}
</section>

<section>
  <h2>6. Subgroup Metrics</h2>
  <p>
    Subgroup metrics 比較模型在不同條件切片下的表現：
    weather、occasion、style 與 category。
    若 Proposed 在多數 subgroup 的 Hit@10 都高於 Original，
    表示改善不是只集中在單一子群體。
    但若某些 subgroup 的 high-confidence error rate 仍高，
    則代表那些條件下仍存在 reliability risk。
  </p>
  {report_img(IMAGES["subgroup_hit10"], "Subgroup Hit@10")}
  {report_img(IMAGES["subgroup_hce"], "Subgroup High-confidence Error Rate")}
  {report_img(IMAGES["subgroup_delta"], "Subgroup ΔHit@10")}
</section>

<section>
  <h2>7. High-confidence Error Analysis</h2>
  <p>
    以下三個案例皆來自 Proposed model。
    它們的共同特徵是 confidence 高於 <b>{HIGH_CONF_THRESHOLD}</b>，
    但正確答案沒有進入 Top-10。
    這類錯誤比一般錯誤更重要，因為使用者更可能相信模型輸出。
  </p>
  {report_img(IMAGES["hce_cases"], "High-confidence Error Examples")}
  {case_report_html}
</section>

<section>
  <h2>8. Possible Error Sources and Fixes</h2>
  <h3>可能錯誤來源</h3>
  <ol>
    <li>
      <b>Confidence calibration 不佳：</b>
      模型 confidence 明顯高於實際 Hit@10 accuracy，造成 over-confidence。
    </li>
    <li>
      <b>語意 shortcut：</b>
      模型可能依賴部分風格詞、類別資訊或視覺相似度，但沒有完整理解 partial outfit 與 target item 的搭配關係。
    </li>
    <li>
      <b>候選池高度相似：</b>
      特別是 accessory-led item，候選項可能外觀相似，使模型容易將不正確 item 排在前面。
    </li>
  </ol>

  <h3>修正策略</h3>
  <ol>
    <li>
      <b>Confidence calibration：</b>
      使用 validation set 進行 temperature scaling 或 isotonic calibration，使 confidence 更接近實際正確率。
    </li>
    <li>
      <b>Reject option / uncertainty warning：</b>
      當校準後 confidence 不足或 top candidates 差距不穩定時，系統不要直接給單一推薦，而是提示使用者需要確認。
    </li>
    <li>
      <b>強化弱 subgroup：</b>
      針對 accessory-led 與 high-confidence mismatch 案例加入 hard negative training、category-aware reranking 或補充更多相似候選資料。
    </li>
  </ol>
</section>

<section>
  <h2>9. Real-world Risk Assessment</h2>
  <p>
    若這些錯誤出現在一般穿搭推薦情境，通常不屬於醫療、金融或司法等高風險決策，
    因此整體可視為 <b>低到中度風險</b>。
  </p>
  <p>
    不過 high-confidence error 不能完全視為無害。
    因為模型看起來很有信心，使用者可能更容易相信錯誤推薦。
    若錯誤發生在面試、正式會議、婚禮、極端天氣或長時間戶外活動，
    可能造成場合不合宜、穿著不舒適，甚至影響使用者對系統的信任。
  </p>
  <div class="alert alert-warning">
    <b>最終建議：</b>Proposed model 相較 Original 有 performance improvement，但仍不應直接部署為完全可信的自動推薦系統；
    真實使用時需要加入 confidence calibration、uncertainty warning 與人工確認機制。
  </div>
</section>

</div>
</body>
</html>
"""

with open(REPORT_HTML, "w", encoding="utf-8") as f:
    f.write(report_html)


# ============================================================
# 8. Dashboard HTML (包含優化的 Modal 與色彩升級)
# ============================================================

case_cards_html = ""

for case in case_cards:
    case_cards_html += f"""
    <details class="case-detail">
      <summary>{escape(case["title"])} <span class="text-muted">| {case["summary"]}</span></summary>
      <div class="case-body">
        <p>{case["analysis"]}</p>
        <pre>{escape(json.dumps(case["log"], ensure_ascii=False, indent=2))}</pre>
      </div>
    </details>
    """

dashboard_html = f"""
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<title>Reliability Dashboard: Original vs Proposed</title>
<style>
:root {{
  --bg: #f1f5f9;
  --card: #ffffff;
  --text-main: #334155;
  --text-heading: #0f172a;
  --text-muted: #64748b;
  --border: #e2e8f0;
  
  --primary: #3b82f6;
  --primary-hover: #2563eb;
  --blue-bg: #eff6ff;
  --green-bg: #f0fdf4;
  --green-border: #10b981;
  --yellow-bg: #fffbeb;
  --yellow-border: #f59e0b;
  --red-bg: #fef2f2;
  --red-border: #ef4444;
}}
@media print {{
  * {{ -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important; }}
  body {{ background: #ffffff !important; }}
}}

body {{
  margin: 0; background: var(--bg); color: var(--text-main);
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
  -webkit-font-smoothing: antialiased;
}}
.container {{ max-width: 1240px; margin: 0 auto; padding: 40px 24px 80px; }}

.header {{
  background: var(--card); border: 1px solid var(--border); border-radius: 20px;
  padding: 40px; margin-bottom: 24px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.02);
  text-align: center;
}}
.header h1 {{ margin: 0 0 12px; font-size: 32px; color: var(--text-heading); font-weight: 700; }}
.header p {{ margin: 0 auto; color: var(--text-muted); font-size: 16px; max-width: 700px; line-height: 1.6; }}

.top-actions {{ display: flex; gap: 12px; justify-content: center; margin-top: 24px; }}
.btn {{
  border: none; border-radius: 10px; padding: 12px 20px; font-size: 15px; font-weight: 600;
  cursor: pointer; text-decoration: none; transition: all 0.2s;
}}
.btn-primary {{ background: var(--primary); color: white; box-shadow: 0 4px 6px -1px rgba(59,130,246,0.3); }}
.btn-primary:hover {{ background: var(--primary-hover); transform: translateY(-1px); }}
.btn-secondary {{ background: white; color: var(--text-main); border: 1px solid var(--border); }}
.btn-secondary:hover {{ background: #f8fafc; border-color: #cbd5e1; }}

/* 數據卡片區 */
.summary-grid {{
  display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 20px; margin: 24px 0;
}}
.metric {{
  background: var(--card); border-radius: 16px; padding: 20px 24px;
  border: 1px solid var(--border); border-left: 4px solid var(--primary);
  box-shadow: 0 4px 6px -1px rgba(0,0,0,0.02);
}}
.metric-label {{ color: var(--text-muted); font-size: 14px; font-weight: 500; }}
.metric-value {{ font-size: 28px; font-weight: 700; color: var(--text-heading); margin: 8px 0 4px; }}
.metric-value.positive {{ color: var(--green-border); }}
.metric-value.negative {{ color: var(--red-border); }}
.metric-note {{ color: var(--text-muted); font-size: 13px; }}

.card {{
  background: var(--card); border-radius: 20px; padding: 32px; margin: 24px 0;
  border: 1px solid var(--border); box-shadow: 0 4px 6px -1px rgba(0,0,0,0.02);
}}
.card h2 {{
  margin: 0 0 20px; font-size: 20px; color: var(--text-heading);
  display: flex; align-items: center; gap: 10px;
}}
.card h2::before {{
  content: ""; display: block; width: 4px; height: 20px; background: var(--primary); border-radius: 4px;
}}

/* 圖表畫廊 */
.gallery {{
  display: grid; grid-template-columns: repeat(auto-fill, minmax(320px, 1fr)); gap: 24px;
}}
.figure-card {{
  background: var(--card); border-radius: 16px; overflow: hidden;
  border: 1px solid var(--border); cursor: pointer;
  transition: all 0.2s ease; display: flex; flex-direction: column;
}}
.figure-card:hover {{
  transform: translateY(-4px); box-shadow: 0 12px 24px -8px rgba(0,0,0,0.1); border-color: var(--primary);
}}
.figure-img-wrapper {{
  height: 200px; background: #f8fafc; border-bottom: 1px solid var(--border);
  display: flex; align-items: center; justify-content: center; overflow: hidden;
}}
.figure-img-wrapper img {{
  width: 100%; height: 100%; object-fit: contain; padding: 10px; box-sizing: border-box;
}}
.figure-card-body {{ padding: 20px; flex: 1; display: flex; flex-direction: column; }}
.figure-card h3 {{ margin: 0 0 8px; font-size: 16px; color: var(--text-heading); }}
.figure-card p {{ margin: 0 0 12px; color: var(--text-muted); font-size: 14px; line-height: 1.5; flex: 1; }}
.badge {{
  align-self: flex-start; display: inline-block; font-size: 12px; font-weight: 600;
  background: var(--blue-bg); color: var(--primary-hover); border-radius: 6px; padding: 4px 10px;
}}
.badge-light {{ background: #f1f5f9; color: var(--text-muted); font-weight: normal; }}

/* 檢查表與說明區 */
.checklist {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(400px, 1fr)); gap: 16px; }}
.check-item {{
  background: var(--blue-bg); border-left: 4px solid var(--primary);
  padding: 16px 20px; border-radius: 12px; font-size: 14.5px; line-height: 1.6;
}}
.check-item.success {{ background: var(--green-bg); border-left-color: var(--green-border); }}
.risk-box {{
  background: var(--red-bg); border-left: 4px solid var(--red-border);
  border-radius: 12px; padding: 20px; line-height: 1.7; font-size: 15px;
}}

/* 案例折疊區 */
.case-detail {{
  background: #f8fafc; border: 1px solid var(--border); border-radius: 12px;
  margin: 12px 0; padding: 16px 20px;
}}
.case-detail summary {{ cursor: pointer; font-weight: 600; color: var(--text-heading); outline: none; }}
.case-detail summary .text-muted {{ font-weight: 400; color: var(--text-muted); font-size: 14px; }}
.case-body {{ margin-top: 16px; line-height: 1.7; font-size: 14.5px; }}
pre {{ background: #1e293b; color: #f8fafc; border-radius: 10px; padding: 16px; overflow-x: auto; font-size: 13px; }}

/* Modal (左圖右文雙欄佈局) */
.modal-overlay {{
  display: none; position: fixed; z-index: 1000; inset: 0;
  background: rgba(15, 23, 42, 0.7); backdrop-filter: blur(4px);
  padding: 40px 20px; overflow-y: auto;
  opacity: 0; transition: opacity 0.3s;
}}
.modal-overlay.show {{ opacity: 1; }}
.modal-content {{
  background: var(--card); border-radius: 20px; max-width: 1200px; margin: 0 auto;
  box-shadow: 0 25px 50px -12px rgba(0,0,0,0.25); overflow: hidden;
  transform: translateY(20px); transition: transform 0.3s ease;
  display: flex; flex-direction: column; min-height: 60vh;
}}
.modal-overlay.show .modal-content {{ transform: translateY(0); }}

.modal-header {{
  padding: 24px 32px; border-bottom: 1px solid var(--border);
  display: flex; justify-content: space-between; align-items: center;
}}
.modal-header h2 {{ margin: 0; font-size: 22px; color: var(--text-heading); }}
.modal-header p {{ margin: 6px 0 0 0; color: var(--text-muted); font-size: 14px; }}
.close-btn {{
  background: none; border: none; width: 40px; height: 40px; border-radius: 50%;
  font-size: 24px; color: var(--text-muted); cursor: pointer; transition: all 0.2s;
  display: flex; align-items: center; justify-content: center;
}}
.close-btn:hover {{ background: #f1f5f9; color: var(--text-heading); }}

.modal-body {{
  display: grid; grid-template-columns: 1.2fr 0.8fr; gap: 32px; padding: 32px; align-items: start;
}}
.modal-img-col img {{ width: 100%; border: 1px solid var(--border); border-radius: 12px; display: block; }}
.modal-text-col {{ font-size: 15px; line-height: 1.7; color: var(--text-main); }}
.modal-text-col h3 {{ margin: 0 0 12px 0; font-size: 16px; color: var(--text-heading); border-bottom: 2px solid var(--border); padding-bottom: 8px; }}
.takeaway-box {{ background: var(--blue-bg); border-left: 4px solid var(--primary); padding: 16px; border-radius: 12px; margin-bottom: 24px; }}

@media (max-width: 900px) {{
  .modal-body {{ grid-template-columns: 1fr; }}
}}
</style>
</head>

<body>
<div class="container">

  <div class="header">
    <h1>Reliability Dashboard</h1>
    <p>透過視覺化圖表與數據切片，深度比較 <b>Original</b> 與 <b>Proposed</b> 模型的 Retrieval Performance 與 Confidence Calibration。</p>
    <div class="top-actions">
      <a class="btn btn-primary" href="reliability_report.html" target="_blank">📄 閱讀完整報告書</a>
      <button class="btn btn-secondary" onclick="scrollToGallery()">📊 瀏覽圖表畫廊</button>
    </div>
  </div>

  <div class="summary-grid">
    <div class="metric">
      <div class="metric-label">Hit@10 improvement</div>
      <div class="metric-value positive">{pp(metric_delta_hit10)}</div>
      <div class="metric-note">Proposed {pct(prop["hit10"])} vs Original {pct(base["hit10"])}</div>
    </div>
    <div class="metric" style="border-left-color: #8b5cf6;">
      <div class="metric-label">Median Rank change</div>
      <div class="metric-value">{metric_delta_rank:+.0f}</div>
      <div class="metric-note">負值代表 Proposed 排名更前面</div>
    </div>
    <div class="metric" style="border-left-color: #f59e0b;">
      <div class="metric-label">ECE change</div>
      <div class="metric-value">{metric_delta_ece:+.3f}</div>
      <div class="metric-note">負值代表 calibration 變好</div>
    </div>
    <div class="metric" style="border-left-color: #ef4444;">
      <div class="metric-label">High-conf Error change</div>
      <div class="metric-value negative">{pp(metric_delta_hce)}</div>
      <div class="metric-note">負值代表高信心錯誤下降</div>
    </div>
  </div>

  <div class="card">
    <h2>作業要求檢查列表</h2>
    <div class="checklist">
      <div class="check-item success"><b>✅ 已報告 Performance</b><br>見圖卡：Performance Comparison。</div>
      <div class="check-item success"><b>✅ 已報告 Confidence Distribution</b><br>見圖卡：Confidence Distribution。</div>
      <div class="check-item success"><b>✅ 已報告 Error Slicing</b><br>見圖卡：Error Slicing。</div>
      <div class="check-item success"><b>✅ 已報告 Subgroup Metrics</b><br>見圖卡：Subgroup Hit@10 / High-conf Error。</div>
      <div class="check-item success"><b>✅ 已分析 High-confidence error</b><br>見下方「3 個極端錯誤案例分析」。</div>
      <div class="check-item success"><b>✅ 提出來源、策略與風險判斷</b><br>詳見底部區塊與輸出之報告書。</div>
    </div>
  </div>

  <div class="card" id="gallery-section">
    <h2>圖表畫廊 (點選圖片查看詳解)</h2>
    <div class="gallery" id="gallery"></div>
  </div>

  <div class="card">
    <h2>3 個極端 High-confidence Error 分析</h2>
    <p style="margin-top:0; color:var(--text-muted); font-size:14.5px;">
      以下案例皆來自 Proposed model，共同條件為 <b>Confidence ≥ {HIGH_CONF_THRESHOLD}</b> 但正確答案未進入 Top-10。
    </p>
    {case_cards_html}
  </div>

  <div class="card">
    <h2>可能錯誤來源與修正策略</h2>
    <div class="checklist">
      <div class="check-item" style="background:#fffbeb; border-color:#f59e0b;">
        <b>⚠️ 錯誤來源 1：Confidence Calibration 不佳</b><br>
        模型 confidence 明顯高於實際 Hit@10 accuracy，造成 Over-confidence 陷阱。
      </div>
      <div class="check-item" style="background:#f0fdf4; border-color:#10b981;">
        <b>💡 修正策略 1：Calibration Method</b><br>
        使用 validation set 做 Temperature Scaling 或 Isotonic Calibration。
      </div>
      <div class="check-item" style="background:#fffbeb; border-color:#f59e0b;">
        <b>⚠️ 錯誤來源 2：語意 Shortcut / 候選池過度相似</b><br>
        模型可能依賴局部風格詞、類別特徵或視覺相似度，沒有完整理解全域搭配關係。
      </div>
      <div class="check-item" style="background:#f0fdf4; border-color:#10b981;">
        <b>💡 修正策略 2：Hard Negative + Reranking</b><br>
        對 Accessory-led 與 Semantic mismatch 案例加強 Hard negative training 或 Category-aware reranking。
      </div>
    </div>
  </div>

  <div class="card">
    <h2>真實使用情境風險判斷</h2>
    <div class="risk-box">
      若錯誤出現在一般穿搭推薦情境，通常不屬於醫療、金融或司法等高風險決策，因此整體是 <b>低到中度風險</b>。<br><br>
      但 High-confidence error 仍需要高度警惕，因為使用者極可能因模型「看起來很有把握」而盲目相信錯誤推薦。
      若發生在正式場合、面試、婚禮、極端天氣等情境，可能造成場合不合宜或穿著不舒適。<br><br>
      <b>部署結論：</b> Proposed 雖然相較 Original 準確率有改善，但真實部署時仍應在後端加入 <b>Confidence Calibration</b>，並在 UI 上加上 <b>Uncertainty Warning (條件不足提示)</b> 與人工確認機制。
    </div>
  </div>

</div>

<!-- 互動彈窗結構 -->
<div class="modal-overlay" id="modal" onclick="closeModal(event)">
  <div class="modal-content" onclick="event.stopPropagation()">
    <div class="modal-header">
      <div>
        <h2 id="modal-title"></h2>
        <p id="modal-subtitle"></p>
      </div>
      <button class="close-btn" onclick="closeModal(event)">&times;</button>
    </div>
    <div class="modal-body">
      <div class="modal-img-col">
        <img id="modal-img" src="" alt="Chart">
      </div>
      <div class="modal-text-col">
        <h3>指標解讀重點 (Takeaway)</h3>
        <div class="takeaway-box" id="modal-takeaway"></div>
        <h3>圖表詳細說明</h3>
        <div id="modal-detail"></div>
      </div>
    </div>
  </div>
</div>

<script>
const cards = {image_cards_json};

function renderGallery() {{
  const gallery = document.getElementById("gallery");
  gallery.innerHTML = "";

  cards.forEach((card, idx) => {{
    const div = document.createElement("div");
    div.className = "figure-card";
    div.onclick = () => openModal(idx);

    div.innerHTML = `
      <div class="figure-img-wrapper">
        <img src="${{card.img}}" alt="${{card.title}}">
      </div>
      <div class="figure-card-body">
        <h3>${{card.title}}</h3>
        <p>${{card.subtitle}}</p>
        <span class="badge">${{card.required}}</span>
      </div>
    `;
    gallery.appendChild(div);
  }});
}}

function openModal(idx) {{
  const card = cards[idx];
  document.getElementById("modal-title").innerText = card.title;
  document.getElementById("modal-subtitle").innerText = card.subtitle;
  document.getElementById("modal-img").src = card.img;
  document.getElementById("modal-takeaway").innerHTML = card.takeaway;
  document.getElementById("modal-detail").innerHTML = card.detail;

  const modal = document.getElementById("modal");
  modal.style.display = "block";
  setTimeout(() => modal.classList.add('show'), 10);
  document.body.style.overflow = 'hidden';
}}

function closeModal(e) {{
  if(e) e.stopPropagation();
  const modal = document.getElementById("modal");
  modal.classList.remove('show');
  setTimeout(() => {{
    modal.style.display = "none";
    document.body.style.overflow = '';
  }}, 300);
}}

document.addEventListener('keydown', function(event) {{
  if (event.key === "Escape" && document.getElementById("modal").style.display === 'block') {{
    closeModal();
  }}
}});

function scrollToGallery() {{
  document.getElementById("gallery-section").scrollIntoView({{ behavior: "smooth" }});
}}

renderGallery();
</script>

</body>
</html>
"""

with open(DASHBOARD_HTML, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("已輸出 dashboard：", DASHBOARD_HTML)
print("已輸出報告書：", REPORT_HTML)
print("Dashboard 內的「📄 閱讀完整報告書」按鍵會開啟 reliability_report.html")
print("提示：點擊報告書中的「列印 / 另存 PDF」，即可匯出完美的彩色 PDF 檔案。")

已輸出 dashboard： ./reliability_outputs/compare/reliability_dashboard.html
已輸出報告書： ./reliability_outputs/compare/reliability_report.html
Dashboard 內的「📄 閱讀完整報告書」按鍵會開啟 reliability_report.html
提示：點擊報告書中的「列印 / 另存 PDF」，即可匯出完美的彩色 PDF 檔案。
